**Project**: GTM Rubric

**Description**: Using gemini AI, finding the root cause of the case with the help of transcripts between customer/seller & support agents.

**Last Updated on**: 20-01-2026

**Contributor**: *Elavala Srinivasa Reddy (elreddy@google.com)*

**Change logs**: including dynamic agent pipelines for each framework

In [ ]:
# @title 1. Installation & Imports
import warnings
warnings.filterwarnings('ignore')

!pip install --upgrade google-cloud-aiplatform google-cloud-logging gspread_pandas tenacity nest_asyncio --quiet

import asyncio
import nest_asyncio
import datetime, pytz
import json
import re
import time
import pandas as pd
import numpy as np
import vertexai

from vertexai.generative_models import GenerativeModel, Part
from vertexai.preview import caching

from google.oauth2.credentials import Credentials
from gspread_pandas import Spread
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.api_core.exceptions import ResourceExhausted, ServiceUnavailable, InternalServerError
from tqdm.asyncio import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

nest_asyncio.apply()

In [ ]:
# @title 2. Configuration

# --- GLOBAL CONFIG ---
JOB_CONFIG = {
    "job_id_logging": "GTM_Rubrics_RCA",
    "agent_model": "gemini-2.5-pro",
    "model_temperature": 0.3, # Low temp for strict formatting
    "project_id": "gtm-cloud-helpdesk",
    "location": "us-central1",
    "batch_size": 50,
    "save_chunk_size": 50
}

# --- GOOGLE SHEETS & DOCS ---
IO_CONFIG = {
    #"spreadsheet_id": "1tbrQxJbjRLEn6yKB7djXsPfkFMKWHICKTy-X8Y5KzRY",
    #"spreadsheet_id": "1ExMsleXwlAV6dVBR4u6936wbSz1-xRTrl6xuRs2mgBM",
    #"spreadsheet_id": "1G3p6oKcsp71YP1pI180gow1XBn47KpXA1zZQa6ej154",
    "spreadsheet_id": "1Lmo5laSelj8Yp-ANjuZ_cQVnVM3W9XB6mNRt_J94juM",
    #"input_sheet_name": "Raw Dump_copy",
    "input_sheet_name": "Cases for Summary",
    "output_sheet_name": "RCA_Analysis_Output_1"
}

DOC_MAPPING = {
    "SOP_Guide": "1LJPbOEi7eUB21ndEFx8NFLc3QRBxjF_lVf1lusRIvkw",
    "Terms_Conditions": "14uSCmn0OZB9x3Mm2Zi6HsrVI5I181mpx_ESlA2kQmak",
    "Plan_Summary": "1A3lwFGFAmiNHCTsKg2lNhn74eC0tRGYg6vaPzdyUr9g",
}

SHEET_MAPPING = {
    "Do_Not_Contact_Data": {
        "file_id": "1J0VyGVmWjoZBbyk7Ra9vl2Bh3mdG6ZCMADkCzBhUh6Y",
        "tabs": ["Do NOT tag"]
    }
}

# --- COLUMN MAPPING ---
COL_MAPPING = {
    "case_number": "Case_Number",
    "case_status": "Case_Status",
    "transcript": "Concatenated_Summary",
    "reopen_counter": "Reopen_Counter",
    "aging_days": "Case_Aging_in_Days",
    "is_escalated": "Escalated",
    "survey_result": "Survey_Result",
    "created_date": "Created_date",
    "closed_date": "Closed_date",
    "reopen_date": "Modified_date",
    "issue_category": "Issue_Category",
    "case_subject": "Case_Subject",
    "case_description": "Case_Description",
    "case_history": "Case History ",
    "vector_body": "Vector_Body",

    # Output Columns
    "out_reopen": "Reopen_RCA_Output",
    "out_ttr": "TTR_RCA_Output",
    "out_escalation": "Escalation_RCA_Output",
    "out_dsat": "DSAT_RCA_Output",
    "out_quality": "Quality_RCA_Output",
    "out_workflow": "Workflow_RCA_Output",
    "out_score": "L3_Score",  # New
    "out_grade": "Grade",     # New
    "out_timestamp": "RCA_Generated_At"
}

OUTPUT_COLS_TO_SAVE = [
    COL_MAPPING["case_number"], COL_MAPPING["case_status"], COL_MAPPING["created_date"],
    COL_MAPPING["issue_category"],
    COL_MAPPING["out_reopen"], COL_MAPPING["out_ttr"],
    COL_MAPPING["out_escalation"], COL_MAPPING["out_dsat"],
    COL_MAPPING["out_quality"],COL_MAPPING["out_workflow"],
    COL_MAPPING["out_score"],  # New
    COL_MAPPING["out_grade"],  # New
    COL_MAPPING["out_timestamp"]
]

# --- SCORING CONFIGURATION ---
SCORING_DICT = {
    # Quality L3s (Examples - adjust scores as needed)
    "Did not seek confirmation if all questions were resolved": 10,
    "Did not use language correctly (grammar, spelling, syntax)": 15,
    "Did not structure response": 20,
    "Did not respond to the initial query in a timely manner": 25,
    "Missed expectations for follow up with the user": 30,
    "Did not respond with appropriate level of empathy": 15,
    "Did not answer all explicit / implicit questions or demonstrate comprehension of the issue": 20,
    "Did not tailor a solution to user's needs": 25,
    "Did not exude ownership (Unprofessional tone, Delegation issues etc.)": 30,
    "Asked for information repeatedly or unnecessarily": 10,
    "Identified a wrong root cause": 20,
    "Provided an inaccurate solution": 25,
    "Did not correct user's misunderstanding": 15,
    "Created confusion by providing unnecessary information": 10,

    # Default score for unmapped L3s, or "None"/"Error"
    "None": 0,
    "Error": 0
}

In [ ]:
# @title 3. Cloud Logging

# --- CLOUD LOGGING SETUP ---
from google.cloud import logging as cloud_logging
import logging

# 1. Generate Unique Job ID (Task Name + Time)
curr_time = datetime.datetime.now(pytz.timezone('Asia/Kolkata')).strftime('%Y_%m_%d_%H_%M_%S')
JOB_ID = f"{JOB_CONFIG['job_id_logging']}_{curr_time}"

print(f"🆔 Job ID: {JOB_ID}")

try:
    # 2. Initialize Cloud Logging Client with your Creds
    log_client = cloud_logging.Client()

    # 3. Connect to Python's standard logging handler
    # This captures all standard logging events and sends them to Cloud Logging
    log_client.setup_logging()

    # 4. Create a named logger for this specific run
    logger = logging.getLogger(JOB_ID)
    logger.setLevel(logging.INFO)

    logger.info(f"🚀 Job Started. ID: {JOB_ID}")
    print("✅ Cloud Logging Configured.")

except Exception as e:
    # Fallback if Cloud Logging fails (e.g., permissions)
    print(f" ⚠️ Cloud Logging Init Failed: {e}")
    # Create a basic local logger so code doesn't break
    logger = logging.getLogger("Local_Fallback")
    logger.setLevel(logging.INFO)

INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:🚀 Job Started. ID: GTM_Rubrics_RCA_2026_03_10_13_03_32


🆔 Job ID: GTM_Rubrics_RCA_2026_03_10_13_03_32
✅ Cloud Logging Configured.


In [ ]:
# @title 4. Hierarchy Configuration

# --- HIERARCHY CONFIGURATION ---
HIERARCHY_CONFIG = {
    "Reopen": {
        "Invalid Reopen": {
            "Invalid Reopen": [
                "Thank you email or Request closure"
            ]
        },
        "People Gap": {
            "Accuracy": [
                "Identified a wrong root cause",
                "Provided an inaccurate solution"
            ],
            "Communication Skills": [
                "Asked for information repeatedly or unnecessarily",
                "Poor arituculation of the solution",
                "Did not structure response",
                "Did not exude ownership",
                "Did not respond with appropriate level of empathy"
            ],
            "Completeness": [
                "Did not answer all explicit / implicit questions or demonstrate comprehension of the issue ",
                "Did not seek confirmation if all questions were resolved",
                "Did not correct user's misunderstanding "
            ],
            "Relevance": [
                "Created confusion by providing unnecessary information",
                "Did not tailor a solution to user's needs"
            ],
            "Responsiveness": [
                "Did not respond to the initial query in a timely manner(24 Hours) ",
                "Missed expectations for follow up with the user"
            ]
        },
        "Process Gap": {
            "Workflow Complexities": [
                "Approvals/Exceptions Required",
                "Complex Processes e.g. Quarter close guidelines",
                "Bulk Requests"
            ],
            "XFn Support Efficacy": [
                "Delayed/dissatisfactory response from dependent teams",
                "Delayed/Inaccurate routing to another XFn support team",
                "Delayed/Inaccurate consult response",
                "Delayed/Inaccurate routing due to complexity of the case"
            ]
        },
        "User Gap": {
            "Missing Information": [
                "User provided incomplete/inaccurate information",
                "WOCA"
            ],
            "Mistakenly opened": [
                "Duplicate ",
                "Blank Reopens"
            ],
            "New Query": [
                "Additional or different Query"
            ],
            "Policies": [
                "User disagrees with the policies/processes/functionalities"
            ],
            "Timeline": [
                "Case closure after the specified deadline"
            ]
        }
    },
    "TTR": {
        "Out of Scope\n": {
            "Out of Scope": [
                "Out of Scope",
                "No Escalations paths exist"
            ]
        },
        "People Gap": {
            "Delay by MSP (Managed Service Provider)": [
                "Low Capacity (Bandwidth Shrinkage)",
                "Delay in case assignment"
            ],
            "In Assign - Associate Delay": [
                "Lack of attention by the associate",
                "Delay in raising consult to SME ",
                "Associate didn't create a child case within stipulated time",
                "Associate took long time to troubleshoot",
                "Wrong case state selected by the associate",
                "Wrong interpretation of seller query",
                "Delay in transferring cases to the XFn team",
                "Weekend/Holiday - Controllable error",
                "Unclear scope"
            ],
            "Reopen - Associate Error": [
                "Inaccurate answer provided by the associate",
                "Incomplete answer provided by the associate",
                "Response sent to the wrong recipient",
                "Escalation - L1 Associate Quality Error"
            ],
            "WOCA/WOCAP - Associate Error": [
                "Insufficient info requested by the associate",
                "Incorrect info requested by the associate",
                "Incorrect approvals requested by the associate"
            ]
        },
        "Process Gap": {
            "Backlogs": [
                "Backlog due to PPH",
                "Backlog due to PPR",
                "Weekend/Holiday - UnControllable"
            ],
            "Case arriving late to the queue": [
                "Delay in transferring cases from XFn other teams",
                "Incorrect routing due to complexity of the case"
            ],
            "Consult Delays": [
                "Delay in raising consult from SME to L2",
                "Delayed consult response from L1.5(SME)",
                "Delayed consult response from L2 (FTEs)",
                "Delayed consult response from XFn Team",
                "Low quality consult response leading to multiple iterations"
            ],
            "Documentation": [
                "Missing Documentation",
                "Low Quality Documentation"
            ],
            "High Incoming Volume": [
                "Delays due to higher incoming volumes than forecasted",
                "Delays due to planned volume increase "
            ],
            "Workflow Complexities": [
                "Approvals/Exceptions Required",
                "Complex Processes",
                "Bulk Requests"
            ]
        },
        "Product/Tools Gap": {
            "Product Bugs": [
                "Automation failure resulting in a delayed manual case",
                "Technical issues with GCBP (ETM, BACS, Roomba, Rainmaker etc)",
                "Bugs leading to delays"
            ],
            "Product Complexity": [
                "Latency issue (need time to reflect changes)"
            ],
            "Product Limitation": [
                "Feature is not Available/doesn't exist"
            ]
        },
        "User Gap": {
            "User Gap [Delay due to Submitter/Seller responses]": [
                "WOCA",
                "WOCAP",
                "New/Follow up Questions",
                "Invalid Reopens",
                "User provided incomplete/inaccurate information",
                "User disagrees with the policies/processes/functionalities"
            ]
        }
    },
    "Escalation":{
        "Invalid Escalations": {
            "Invalid Escalations": [
                "Invalid Requests",
                "Infeasible Requests",
                "No Action required"
            ]
        },
        "Out of Scope": {
            "Out of Scope": [
                "Non Data Quality Changes",
                "No Escalations paths exist"
            ]
        },
        "People Gap": {
            "Accuracy": [
                "Identified a wrong root cause",
                "Provided an inaccurate solution"
            ],
            "Communication Skills": [
                "Asked for information repeatedly or unnecessarily",
                "Did not exude ownership",
                "Did not respond with appropriate level of empathy"
            ],
            "Completeness": [
                "Did not answer all explicit / implicit questions or demonstrate comprehension of the issue",
                "Did not seek confirmation if all questions were resolved",
                "Did not correct user's misunderstanding"
            ],
            "Relevance": [
                "Created confusion by providing unnecessary information",
                "Did not tailor a solution to user's needs"
            ],
            "Responsiveness": [
                "Did not respond to the initial query in a timely manner",
                "Missed expectations for follow up with the user"
            ]
        },
        "Process Gap": {
            "Documentation": [
                "Missing Documentation",
                "Low Quality Documentation"
            ],
            "Vol. spikes/backlogs": [
                "High Incoming Vols than forecast"
            ],
            "Workflow Complexities": [
                "Approvals/Exceptions Required",
                "Complex Processes",
                "Bulk Requests"
            ],
            "XFn Support Efficacy": [
                "Dependency on XFn Team for next steps",
                "Delayed/Inaccurate routing to another XFn support team",
                "Delayed/Inaccurate consult response",
                "Delayed/Inaccurate response from another XFn support team",
                "Delayed/Inaccurate routing due to complexity of the case"
            ]
        },
        "Product/Tools Gap": {
            "Product Bugs": [
                "Bug/Technical issues with GCBP",
                "Bug/Technical issues with internal tools"
            ],
            "Product Complexity": [
                "Latency issue (need time to reflect changes)"
            ],
            "Product Limitation": [
                "Feature is not Available/doesn't exist"
            ]
        },
        "User Gap": {
            "User Gap": [
                "Automated Workflows",
                "Access Limitations",
                "Self Help Enabled",
                "Education",
                "Just in case",
                "WOCA",
                "Expedited Requests",
                "User provided incomplete/inaccurate information",
                "User disagrees with the policies/processes/functionalities",
                "New/Follow up Questions",
                "Duplicate"
            ]
        }
    },
    "DSAT": {
        "Out of Scope": {
            "Out of Scope": [
                "Non Data Quality Changes "
            ]
        },
        "People Gap": {
            "Accuracy": [
                "Identified a wrong root cause ",
                "Provided an inaccurate solution"
            ],
            "Communication Skills": [
                "Asked for information repeatedly or unnecessarily",
                "Did not exude ownership",
                "Did not respond with appropriate level of empathy"
            ],
            "Completeness": [
                "Did not answer all explicit / implicit questions or demonstrate comprehension of the issue ",
                "Did not seek confirmation if all questions were resolved",
                "Did not correct user's misunderstanding "
            ],
            "Relevance": [
                "Created confusion by providing unnecessary information",
                "Did not tailor a solution to user's needs"
            ],
            "Responsiveness": [
                "Did not respond to the initial query in a timely manner",
                "Missed expectations for follow up with the user"
            ]
        },
        "Process Gap ": {
            "Documentation": [
                "Missing Documentation",
                "Low Quality Documentation"
            ],
            "Vol. spikes/backlogs": [
                "High Incoming Vols than forecast"
            ],
            "Workflow Complexities": [
                "Approvals/Exceptions Required",
                "Complex Processes",
                "Bulk Requests"
            ],
            "XFn Support Efficacy": [
                "Dependency on XFn Team for next steps ",
                "Delayed/Inaccurate routing to another XFn support team",
                "Delayed/Inaccurate consult response",
                "Delayed/Inaccurate response from another XFn support team",
                "Delayed/Inaccurate routing due to complexity of the case"
            ]
        },
        "Product/Tools Gap": {
            "Product Bugs": [
                "Bug/Technical issues with GCBP",
                "Bug/Technical issues with internal tools "
            ],
            "Product Limitation": [
                "Feature is not Available/doesn't exist",
                "Latency issue (need time to reflect changes)"
            ]
        },
        "User Gap": {
            "User Gap": [
                "User disagrees with the policies/processes/functionalities",
                "User disagrees with the outcome",
                "User expects unrealistic response time",
                "Wrong survey response submitted accidentally",
                "User request was unclear/inaccurate/incomplete",
                "WOCA"
            ]
        }
    },
    "Quality":{
        "People Gap": {
            "Accuracy": [
            "Identified a wrong root cause ",
            "Provided an inaccurate solution"
            ],
            "Communication Skills": [
            "Asked for information repeatedly or unnecessarily",
            "Did not use language correctly (grammar, spelling, syntax)",
            "Did not structure response",
            "Did not exude ownership (Unprofessional tone, Delegation issues etc.)",
            "Did not respond with appropriate level of empathy"
            ],
            "Completeness": [
            "Did not answer all explicit / implicit questions or demonstrate comprehension of the issue ",
            "Did not seek confirmation if all questions were resolved",
            "Did not correct user's misunderstanding "
            ],
            "Relevance": [
            "Created confusion by providing unnecessary information",
            "Did not tailor a solution to user's needs"
            ],
            "Responsiveness": [
            "Did not respond to the initial query in a timely manner",
            "Missed expectations for follow up with the user"
            ]
        }
    },
    "Workflow" :{
        "People Gap": {
            "Compliance:Data Integrity & Legal Guidelines": [
            "Unauthorized Disclosure of Confidential Information",
            "Requested PII details from the seller in a case",
            "Deceptive tactics for inflating CSAT scores",
            "Process Bypass or System Abuse",
            "Misconduct - Revealing the contents of confidential records,data,documents ,information to unauthorized employees or persons.",
            "Unprofessional Communication or Inappropriate Communication"
            ],
            "Workflow Adherence Error": [
            "Incorrectly captured the case status",
            "Did not capture all relevant case notes (from meeting/chat/case pings) in the consult case",
            "Did not leave a note when un-assigning a case",
            "Incorrectly assigned the case (to FTEs)",
            "Missing vector case hygiene fields",
            "Didnot follow Escalation triaging workflow",
            "Incorrectly took a case offline or replied to the issue reported in a case, from his/her email id",
            "Followed incorrect duplicate (clone) workflow",
            "Didn't send the correct Canned Response",
            "Didn't Offer solution to the case correctly",
            "Did not pitch the survey before closing the case",
            "Tagging people from DNC list",
            "Incorrectly tagged the RSO(s) ",
            "Invalid transfer",
            "Case stalled without progress",
            "Closing case without informing the seller & closing the case when seller has more queries.",
            "Incorrectly handled the re-open ticket"
            ]
        }
    }
}

In [ ]:
# @title 5. Prompts

# --- PROMPT TEMPLATES (NO LABELS, STRICT PIPE) ---
REOPEN_AGENT_1_PROMPT = """
You are a Senior Triage Auditor. Identify the **Single Most Important Reason** for the case status based on the Transcript and Metadata.


   ### INPUT DATA
   - **TRANSCRIPT:** Contains chat history and `<<< SYSTEM LOG: CASE CLOSED ... >>>`.
   - **METADATA:** Status, Created/Closed/Reopen Dates.


   ### PHASE IDENTIFICATION
   1. Find the **Last** `<<< SYSTEM LOG: CASE CLOSED ... >>>` marker.
   2. Analyze the **"Reopen Phase"** (text AFTER this marker).
   3. Identify the **"Reopen Trigger"** (First Customer message in the Reopen Phase).


   ### VALIDATION RULES (Evaluate in Priority Order)
   **Stop immediately** at the first match.


   #### PRIORITY 1: REOPEN VALIDITY (Critical)
   1. **"Blank Reopens"** (Category: Mistakenly opened)
   - **MATCH IF:** Captures cases reactivated from a closed status without new context, questions, or documentation. These "false starts" typically result from accidental clicks or administrative toggles that require no actual agent intervention also applies when users reopen cases without providing instructions, creating a system alert where no action is needed. This usually stems from user navigation errors or administrative updates rather than a genuine request for support.


   2. **"Thank you email or Request closure"** (Category: Invalid Reopen)
   - **MATCH IF:** This classification applies when a user reopens a resolved case solely to express gratitude, acknowledge updates, or grant permission to close. Agents should re-close these tickets immediately, as no further technical work or investigation is required and we can also consider if user say "you can close the case".
   - **IGNORE IF:** The message asks a question or provides new info.


   3. **"Duplicate"** (Category: Mistakenly opened)
   - **MATCH IF:** The agent identifies the reopened inquiry as redundant because the issue is already being addressed under a separate, active Case ID, or closes the thread while referencing a master ticket handling the request..


   #### PRIORITY 2: RESPONSIVENESS (Major)
   4. **"Did not respond to the initial query in a timely manner(24 Hours)"** (Category: Responsiveness)
   - **MATCH IF:** Gap between (Reopen Trigger) and (Agent's Next Response) is > 24 Hours, calculated based on business days (excluding weekends and holidays) and the agent's specific shift schedule.


   5. **"Missed expectations for follow up with the user"** (Category: Responsiveness)
   - **MATCH IF (In Progress):** Agent promised a specific time *during the reopen phase* and missed it, OR failed to update an "In Progress" case within 24 hours.
   - **MATCH IF (WOCA):** This classification applies when a user reopens a case to provide missing information after the WOCA window expired; however, it is also flagged if the Case was in WOCA status but Agent failed to follow up according to WOCA guidelines (typically every 3 business days) before closing.


   #### PRIORITY 3: TIMELINE (Minor)
   6. **"Case closure after the specified deadline"** (Category: Timeline)
   - **MATCH IF:** Reopen Date is > 7 days after Closed Date, or if case is  reopened by the seller after the 3-day automated WOCA.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown.
   Format: `Category | L3 Rule Name | Justification (max 30 words)`


   Example: `Responsiveness | Did not respond to the initial query in a timely manner(24 Hours) | Agent took 28 hours to reply to the reopen message.`
   If no violation, return: `None`
"""


REOPEN_AGENT_2_PROMPT = """
   You are a Communication Quality Coach. Identify the **Primary Soft Skill Failure** during the **Reopen Phase**.


   ### INPUT DATA
   - **TRANSCRIPT:** Contains chat history and `<<< SYSTEM LOG: CASE CLOSED ... >>>`.
   - **METADATA:** Status, Created/Closed/Reopen Dates.


   ### PHASE IDENTIFICATION
   1. Locate `<<< SYSTEM LOG: CASE CLOSED AT ... >>>`.
   2. Analyze Agent behavior **AFTER** this log.


   ### VALIDATION RULES (Evaluate in Priority Order)


   #### PRIORITY 1: RELATIONSHIP BREAKERS (Critical)
   1. **"Did not respond with appropriate level of empathy"** (Category: Communication Skills)
   - **MATCH IF:** Customer is frustrated/anxious about the reopen, and Agent uses a robotic tone or ignores the emotion.However if the AI transcript lacked empathy by not saying "You're welcome" or acknowledging the seller's response this shouldn't be labeled as a people gap. Instead, it should be labeled as an Invalid Reopen because the seller only thanked the agent and confirmed that the case was closed.


   2. **"Did not exude ownership"** (Category: Communication Skills)
   - **MATCH IF:** Agent says "I don't know" without checking, or fails to transfer/tag relevant POCs during the reopen.


   #### PRIORITY 2: RESOLUTION BLOCKERS (Major)
   3. **"Did not answer all explicit / implicit questions or demonstrate comprehension"** (Category: Completeness)
   - **MATCH IF:** The agent ignored a specific question asked in the Reopen Trigger.There are scenarios where the seller give confirmation on gchat to close the case, and later reopens the case. In those'" scenarios this should not be considered as People Gap. We can look for comments like "confirmation over Google chat or as confirmed on gchat"


   4. **"Asked for information repeatedly or unnecessarily"** (Category: Communication Skills)
   - **MATCH IF:** Agent asks for info again that was provided earlier in the chat.
also if the agent failed to read the case description or previous comments, which already contained the necessary information, and instead asked the seller to repeat the details, causing an unnecessary delay.


   5. **"User provided incomplete/inaccurate information"** (Category: Missing Information)
   - **MATCH IF:** Agent correctly asked for info to resolve the reopen, and User failed to provide it.


   #### PRIORITY 3: HYGIENE & FORMATTING (Minor)
   6. **"Did not structure response"** (Category: Communication Skills)
   - **MATCH IF:** The response lacks mandatory branding ("Cloud GTM HelpDesk"), acknowledgment, paraphrasing, or failing to add standard canned responses (e.g., EOQ CR) OR fails to acknowledge the specific issue OR fails to paraphrase the issue to show understanding.




   7. **"Poor arituculation of the solution"** (Category: Communication Skills)
   - **MATCH IF:** Explanation is confusing, has bad grammar, or lacks basic email etiquette.


   8. **"Did not seek confirmation if all questions were resolved"** (Category: Completeness)
   - **MATCH IF:** The analyst closes a support case without first asking the seller if their problem was actually fixed. Instead of verifying that the seller is satisfied, the analyst unilaterally ends the interaction, often leaving the seller with lingering questions or unresolved issues.
This error also involves incorrect case tagging. The analyst misclassified the case by applying an inappropriate tag, such as labeling it as WOCA when it specifically required an IPCR or IPGE designation. Ultimately, it means the case was closed prematurely, labeled incorrectly, and tucked away before the customer had the final word on whether the solution worked.


   9. **"Created confusion by providing unnecessary information"** (Category: Relevance)
   - **MATCH IF:** Agent pasted irrelevant FAQs not helpful to the specific reopen issue.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown.
   Format: `Category | L3 Rule Name | Justification (max 30 words)`


   Example: `Communication Skills | Did not structure response | Agent failed to include 'Cloud GTM HelpDesk' branding and did not paraphrase the customer's reopen issue.`
   If no violation, return: `None`
"""


REOPEN_AGENT_3_PROMPT = """
   You are a Senior Process Auditor. Identify the **Primary Root Cause** of the Reopen using the SOP Context.


   ### INPUT & PHASE
   - **CASE CONTEXT:** Use 'Description' to understand the *Original* Issue.
   - **TRANSCRIPT:** Use `<<< SYSTEM LOG: CASE CLOSED ... >>>` to split Original vs. Reopen.
   - **ANALYSIS:** Did the **Original Solution** fail? (Priority 1). Did the **Reopen Handling** fail? (Priority 2).


   ### VALIDATION RULES (Evaluate in Priority Order)


   #### PRIORITY 1: ACCURACY (Did the Original Solution Fail?)
   1. **"Identified a wrong root cause"** (Category: Accuracy)
   - **MATCH IF:** Agent's diagnosis (Pre-Log) contradicts the SOP Context for the issue described.
   2. **"Provided an inaccurate solution"** (Category: Accuracy)
   - **MATCH IF:** The Original resolution steps were incorrect or forbidden by SOP, causing the reopen.


   #### PRIORITY 2: WORKFLOW (Did the Process Fail?)
   3. **"Approvals/Exceptions Required"** (Category: Workflow Complexities)
   - **MATCH IF:** Reopen caused by denied/delayed RSO/CEC Approval.
   4. **"Complex Processes e.g. Quarter close guidelines"** (Category: Workflow Complexities)
   - **MATCH IF:** Reopen occurred due to Quarter Close Freeze blocking the request and processing delay caused by the quarter closure freeze.
   5. **"Delayed/Inaccurate routing to another XFn support team"** (Category: XFn Support Efficacy)
   - **MATCH IF:** Case bounced between teams *after* the reopen.
   6. **"Delayed/Inaccurate routing due to complexity of the case"** (Category: XFn Support Efficacy)
   - **MATCH IF:** Case unassigned/bounced due to lack of ownership clarity.
   7. **"Delayed/Inaccurate consult response"** (Category: XFn Support Efficacy)
   - **MATCH IF:** Reopen caused by Agent waiting >48 hours for FTE/Consult response.
   8. **"Delayed/dissatisfactory response from dependent teams"** (Category: XFn Support Efficacy)
   - **MATCH IF:** User reopened due to dissatisfaction with a Consult POC's response.


   #### PRIORITY 3: USER & PROCESS GAPS (Informational)
   9. **"Did not correct user's misunderstanding"** (Category: Completeness)
   - **MATCH IF:** Reopen caused by User expecting something contrary to Policy.
   10. **"Did not tailor a solution to user's needs"** (Category: Relevance)
       - **MATCH IF:** Failed to file Buganizer when required.
   11. **"Bulk Requests"** (Category: Workflow Complexities)
       - **MATCH IF:** Seller reopened to add requests, and Agent correctly directed them to the Bulk Trix/Doc process.
   12. **"WOCA"** (Category: Missing Information)
       - **MATCH IF:** Case closed via WOCA but Seller provided info late.
   13. **"User disagrees with the policies/processes/functionalities"** (Category: Policies)
       - **MATCH IF:** Reopen is just the User arguing against T&C.
   14. **"Additional or different Query"** (Category: New Query)
       - **MATCH IF:** The Reopen Message contains a completely NEW question.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown.
   Format: `Category | L3 Rule Name | Justification (max 30 words)`


   Example: `Accuracy | Provided an inaccurate solution | Agent approved quota change without Manager Approval, which contradicts the SOP, causing the reopen.`
   If no violation, return: `None`
"""



TTR_AGENT_1_PROMPT = """

   You are a TTR (Time To Resolve) Timeline Auditor. Identify the **Primary Time-Based Reason** for the delay.


   ### INPUT DATA
   - **TRANSCRIPT:** Timestamps are in PST.
   - **CASE STATUS HISTORY:** Use this to calculate "Assignment Delay" (New -> Assigned) and "WOCA Duration".
   - **METADATA:** Created_Date, Closed_Date (Assume PST).


   ### ANALYSIS LOGIC
   - **Timezone:** All inputs are in **PST**. Apply "Weekend/Holiday" rules directly.
   - **Priority:** Check for "Uncontrollable Factors" (Priority 1) first. If none, check for "Associate Errors" (Priority 2).


   ### VALIDATION RULES (Evaluate in Priority Order)
   **Stop immediately** at the first match.


   #### PRIORITY 1: UNCONTROLLABLE DELAYS (Process Gaps - Valid Excuses)
   1. **"Backlog due to PPH"** (Category: Process Gap)
   - **MATCH IF:** Resolution is prolonged because required actions, such as a billing ID move, coincided with a system freeze (e.g., quarterly freeze), forcing the ticket to remain pending and includes general prioritization notices related to quarter-end deadlines.


   2. **"Backlog due to PPR"** (Category: Process Gap)
   - **MATCH IF:** `Created_Date` is between **25th** of Mar, Jun, Sep, Dec AND **25th** of the following month (Sales Data Freeze).


   3. **"Weekend/Holiday - Controllable error"** (Category: In Assign - Associate Delay)
   - **MATCH IF:** `Created_Date` falls on Sat, Sun, or Friday > 5:00 PM PST **AND** FMR was delayed beyond the next business day.


   4. **"Weekend/Holiday - UnControllable"** (Category: Process Gap)
   - **MATCH IF:** Delay caused purely by the weekend (e.g., Created Fri, Responded Mon) and delays occurred due to weekends or holidays, where the agent followed up with the seller or applied the WOCA rule (Waiting on Customer Action) only after the break.


   #### PRIORITY 2: ASSOCIATE DELAYS (People Gap - Violations)
   5. **"Low Capacity (Bandwidth Shrinkage)"** (Category: Delay by MSP)
   - **MATCH IF:** FMR Gap (Agent's 1st Reply - Created_Date) > **48 Hours** (excluding weekends).


   6. **"Delay in case assignment"** (Category: Delay by MSP)
   - **LOGIC:** Look at **CASE STATUS HISTORY**. Calculate time between "New" and "Assigned".
   - **MATCH IF:** Gap > **48 Hours** (excluding weekends).


   7. **"Associate took long time to troubleshoot"** (Category: In Assign - Associate Delay)
   - **MATCH IF:** Gap between "Seller's Last Response" and "Agent's Next Reply" is > **48 Hours** (excluding weekends).


   8. **"Delay in raising consult to SME"** (Category: In Assign - Associate Delay)
   - **MATCH IF:** Agent says "I need to consult", but the actual update/consult note appears > *24 Hours* later and delays also incur in the internal process of escalating a case from a SME to an L2 analyst.


   9. **"Delayed consult response from L1.5(SME)"** (Category: Consult Delays)
   - **MATCH IF:** Agent raised consult, but SME took > *24 Hours* to respond and delays  in receiving a response from a Level 1.5 SME, often for policy or account-specific clarifications (e.g., confirming a Greenfield account's eligibility)




   10. **"Delayed consult response from L2 (FTEs)"** (Category: Consult Delays)
       - **MATCH IF:** Agent/SME raised consult to FTE, but FTE took > 24 Hours to respond and delays can also be in receiving a response from an L2 analyst or specialized internal FTE resource, often required for policy clarification or claim validation (e.g., territory ownership disputes)


   11. **"Delayed consult response from XFn Team"** (Category: Consult Delays)
       - **MATCH IF:** Agent raised consult to XFn, but XFn took > *24 Hours* to respond and this classification applies when resolution is significantly delayed by dependency on Cross-Functional Teams (e.g., OMPF) for technical validation, bug fixes, or complex calculations. It represents the primary cause of TTR failure, forcing analysts to wait weeks for external partners to complete necessary backend tasks.


   12. **"Delay in transferring cases to the XFn team"** (Category: Case arriving late)
       - **MATCH IF:** Transfer happened > **24 Hours** after the need was identified.


   #### PRIORITY 3: USER DELAYS (User Gap)
   13. **"WOCA"** (Category: User Gap)
       - **MATCH IF:** Seller took > **3 Days** to provide requested information.


   14. **"WOCAP"** (Category: User Gap)
       - **MATCH IF:**Seller took > *3 Days* to acknowledge/reject a data quality change request and delays caused by the "Waiting on Customer/Approver Action" (WOCAP) process, typically when an internal party, such as an approver (e.g., a duplicate account owner), takes too long to approve a request (e.g., a merge or hierarchy change request).


   15. **"User Gap [Delay due to Submitter/Seller responses]"** (Category: User Gap)
       - **MATCH IF:** General delay where Seller took > 48 Hours to reply to any query.


   16. **"Invalid Reopens"** (Category: User Gap)
       - **MATCH IF:** Case was reopened just to say "Thanks", wasting time.

   17. **"Bugs leading to delays"** (Category:Product/Tools Gap)
       - **MATCH IF:**  If the case is pending with engineering team as well due to bug  and also If there are word like engineering team or bug and the status was IPGE or Known seller issue should considered for "Bug/Technical issues with internal tools.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown.
   Format: `Category | L3 Rule Name | Justification (max 30 words)`


   Example: `In Assign - Associate Delay | Associate took long time to troubleshoot | Agent took 52 hours to respond to the seller's screenshot, exceeding the 48h threshold.`
   If no violation, return: `None`
"""


TTR_AGENT_2_PROMPT = """
   You are a TTR Process Auditor. Identify the **Primary Content-Based Reason** for the delay using the SOP Context.


   ### INPUT DATA
   - **CASE CONTEXT:** Subject, Description.
   - **CASE STATUS HISTORY:** Use this to verify if the System Status matches the Agent's actions.
   - **TRANSCRIPT:** Full chat history.


   ### VALIDATION RULES (Evaluate in Priority Order)


   #### PRIORITY 1: PRODUCT & WORKFLOW (Valid Delays)
   1. **"Latency issue"** (Category: Product Complexity)
   - **MATCH IF:** Agent mentions "Latency" AND advises seller to wait **3-5 days** for data reflection.


   2. **"Approvals/Exceptions Required"** (Category: Workflow Complexities)
   - **MATCH IF:** Delay caused by mandatory RSO/CEC/Seller Approval steps (Check History for `WOCAP` status) and delays caused by the "Waiting on Customer/Approver Action" (WOCAP) process, typically when an internal party, such as an approver (e.g., a duplicate account owner), takes too long to approve a request (e.g., a merge or hierarchy change request).

   3. **"Complex Processes"** (Category: Workflow Complexities)
   - **MATCH IF:** Case involves Multiple Requests or PPR disagreements requiring FTE.

   4. **"Bulk Requests"** (Category: Workflow Complexities)
   - **MATCH IF:** Seller raised > 5 records/requests in a single ticket.

   5. **"Feature is not Available/doesn't exist"** (Category: Product Limitation)
   - **MATCH IF:** Agent explicitly states the feature does not exist.


   6. **"Automation failure resulting in a delayed manual case"** (Category: Product Bugs)
   - **MATCH IF:** Agent explicitly mentions "Automation failed" or "BACS/Elixir error".


   #### PRIORITY 2: ASSOCIATE ERRORS (Violations)
   7. **"Inaccurate answer provided by the associate"** (Category: Reopen - Associate Error)
   - **MATCH IF:** Agent provided wrong info/resolution which caused the case to drag on.


   8. **"Wrong interpretation of seller query"** (Category: In Assign - Associate Delay)
   - **MATCH IF:** Agent provided a resolution unrelated to the Seller's `Case Description`.


   9. **"Insufficient info requested by the associate"** (Category: WOCA - Associate Error)
   - **MATCH IF:** Agent asked for Item A, then later asked for Item B (Should have asked A+B together).


   10. **"Incorrect info requested by the associate"** (Category: WOCA - Associate Error)
       - **MATCH IF:** Agent asked for info NOT required by the SOP.


   11. **"Incorrect approvals requested by the associate"** (Category: WOCA - Associate Error)
       - **MATCH IF:** Agent asked for RSO/FSR approval when SOP says it's not needed.


   12. **"Lack of attention by the associate"** (Category: In Assign - Associate Delay)
       - **MATCH IF:** case involves a combination of multiple requests and consultations, alongside system bugs identified by the GE team. Additionally, it encompasses PPR disagreements that necessitate FTE intervention to reach a resolution.


   13. **"Associate didn't create a child case within stipulated time"** (Category: In Assign - Associate Delay)
       - **MATCH IF:** SOP required a Child Case, but Agent failed to create one.


   14. **"Wrong case state selected by the associate"** (Category: In Assign - Associate Delay)
       - **LOGIC:** Compare **Transcript** vs **Case Status History**.
       - **MATCH IF:** Agent is working on the case ("I am checking"), but sets status to `WOCA` or `IPCR` to pause the SLA clock.
       - **MATCH IF:** Agent keeps case `In Progress` when it should be `WOCA` (waiting for user).


   15. **"Response sent to the wrong recipient"** (Category: Reopen - Associate Error)
       - **MATCH IF:** Agent addresses the wrong person (e.g., "Hi Name1" then "Hi Name2") without context of handover.


   16. **"Delay in raising consult from SME to L2"** (Category: Consult Delays)
       - **MATCH IF:**SME failed to escalate to FTE when required and also if delays were incurred in the internal process of escalating a case from a SME to an L2 analyst.


   17. **"Low quality consult response leading to multiple iterations"** (Category: Consult Delays)
       - **MATCH IF:** SME provided a generic response, forcing the Agent to ask again and also if the quality of the consultation response was insufficient, leading to back-and-forth communication with the SME and prolonging the case resolution.


   18. **"Missing/Low Quality Documentation"** (Category: Documentation)
       - **MATCH IF:** Agent states they cannot find the resolution in SOP/T&C or gives generic info due to lack of docs and the system-generated case or existing documentation did not provide sufficient detail for the agent to fully inform the seller (e.g., the system showed a domain association but no other details).


   19. **"Incorrect routing due to complexity of the case"** (Category: Case arriving late)
       - **MATCH IF:** Case bounced due to ambiguity/complexity and cases were incorrectly routed initially due to their complex nature, such as when a request for a Warranty Certificate was transferred between teams (PSS and DQO) before a solution was found.


   #### PRIORITY 3: USER GAPS (Explanations)
   20. **"User disagrees with the policies/processes/functionalities"** (Category: User Gap)
       - **MATCH IF:**Case bounced due to ambiguity/complexity and cases were incorrectly routed initially due to their complex nature, such as when a request for a Warranty Certificate was transferred between teams (PSS and DQO) before a solution was found.


   21. **"New/Follow up Questions"** (Category: User Gap)
       - **MATCH IF:** User asks for update on processed request (Data visible but user can't see), OR asks a NEW question and delays occur when sellers extend the case lifecycle by raising new inquiries or requests after the initial resolution is provided. This scope expansion requires additional investigation into matters distinct from the original issue.






   22. **"User provided incomplete/inaccurate information"** (Category: User Gap)
       - **MATCH IF:** Agent asked for specific info and User failed to provide it.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown.
   Format: `Category | L3 Rule Name | Justification (max 30 words)`


   Example: `In Assign - Associate Delay | Wrong case state selected by the associate | Transcript shows agent investigating, but History shows status was changed to WOCA.`
   If no violation, return: `None`
"""



DSAT_AGENT_1_PROMPT = """

You are the DSAT Conversational Auditor. Your task is to analyze the provided labeled transcript and initial case context to identify the single **BEST MATCH** violation of the agent's communication and completeness standards. You must **ONLY** use the text of the **Transcript, Case_Subject, and Case_Description** for your validation.

   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules below and select the single L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **OUTPUT FORMAT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   ### INPUT DATA
   - **Case_Subject**: The subject line of the case.
   - **Case_Description**: The initial problem statement from the seller.
   - **Case_History**: The historical log of the case (for context on previous actions).
   - **Transcript**: The conversation between the seller and the support agent.


   ### VALIDATION RULES (Check these 6 specific L3s - No Metadata/RAG required)


   #### Completeness (Category: Completeness)
   1. **"Did not answer all explicit / implicit questions or demonstrate comprehension of the issue "**
   - VIOLATION IF: Agent provided partial resolution, missed detailed explanation, OR failed to answer a follow-up query.
   2. **"Did not seek confirmation if all questions were resolved"**
   - VIOLATION IF: Agent prioritized "closing the ticket" over "solving the problem" by:
        a) Closing the case without seller consent or insisting on closure despite an explicit request to keep it open.
        b) Prematurely ending the interaction due to misuse of system automation (e.g., failing to update a WOCA tag to IPCR/IPGE, leading to an auto-closure before resolution).
   3. **"Did not correct user's misunderstanding "**
   - VIOLATION IF: Agent failed to handle a seller's objection OR failed to correct a factual error made by the seller (e.g., incorrect routing name).


   #### Communication Skills (Category: Communication Skills)
   4. **"Did not respond with appropriate level of empathy"**
   - VIOLATION IF: Agent's language showed no appropriate level of empathy (e.g., in case of no solution/out of scope).


   #### User Gap (Category: User Gap)
   5. **"User disagrees with the outcome"**
   - VIOLATION IF: User explicitly states disagreement with the final outcome (e.g., request being rejected) despite understanding the T&Cs.
   6. **"User expects unrealistic response time"**
   - VIOLATION IF: User explicitly demands a quicker turnaround time than is standard (e.g., response on weekends, quicker fix for a bug).


   ### OUTPUT FORMAT (JSON ONLY)
   [Category of the L3] | [Exact L3 Rule Name] | [Brief justification (max 30 words)]
"""


DSAT_AGENT_2_PROMPT = """
   You are the DSAT Performance Auditor. Your task is to validate L3 rules using the provided case metadata and system information. You must **CROSS-REFERENCE** the transcript, Case_Subject, and Case_Description with the **METADATA** for validation.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules below and select the single L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data. Prioritize System/Bug issues and SLA misses.
   2. **OUTPUT FORMAT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   ### INPUT DATA
   - **Case_Subject**: The subject line of the case.
   - **Case_Description**: The initial problem statement from the seller.
   - **Case_History**: The historical log of the case (for context on previous actions).
   - **Transcript**: The conversation between the seller and the support agent.
   - **Metadata**: The entire case metadata (CRITICAL: Created_date, Bug_Id, Onwer_Change_Counter).


   ### VALIDATION RULES (Check these 11 specific L3s - Requires Metadata)


   #### Communication Skills (Category: Communication Skills)
   1. **"Asked for information repeatedly or unnecessarily"**
   - VIOLATION IF: Agent asked seller to repeat details which were already present in the **Case_Description** or **Case History/Metadata**, causing unnecessary delay..
   2. **"Did not exude ownership"**
   - VIOLATION IF: Agent was OOO and the lead didn't transfer the case (check owner changes), OR Agent closed the case without checking if the seller is OOO.


   #### Responsiveness (Category: Responsiveness)
   3. **"Did not respond to the initial query in a timely manner"**
   - VIOLATION IF: Agent's first response time (calculated from metadata) exceeds the defined SLA (e.g., 24 hours).
   4. **"Missed expectations for follow up with the user"**
   - VIOLATION IF: Agent failed to follow up on the case in a timely manner OR failed to set expectations (check against WOCA/IPCR/IPGE timelines).


   #### Product/Tools Gap (Category: Product Limitation/Bugs)
   5. **"Feature is not Available/doesn't exist"**
   - VIOLATION IF: The issue is confirmed to be a missing product feature (e.g., user asking to remove partner from opportunity, which is not possible).
   6. **"Latency issue (need time to reflect changes)"**
   - VIOLATION IF: The issue is confirmed to be a time delay for changes to reflect in downstream systems (e.g., DQ move reflection).
   7. **"Bug/Technical issues with GCBP"**
   - VIOLATION IF: Case Metadata (`Bug_Id`/`Bug_Title`) confirms a technical issue with GCBP is the root cause.
   8. **"Bug/Technical issues with internal tools "**
   - VIOLATION IF: Case Metadata (`Bug_Id`/`Bug_Title`) confirms an internal tool bug (e.g., Rainmaker, Vector) resulting in system errors/data mismatch, **OR** the **issue required agent escalation to the engineering team/SME workaround.** ,**OR**  if the case is pending with engineering team as well due to bug, **OR**  If  the status was IPGE or Known seller issue should considered for "Bug/Technical issues with internal tools"





   #### User Gap (Category: User Gap)
   9. **"Wrong survey response submitted accidentally"**
   - VIOLATION IF: Negative CSAT is due to user error rather than service failure, specifically *Misattributed Feedback* (review intended for a different agent/case interaction) or *Rating Mismatch/Misclick* (blatant contradiction between low overall rating and high specific scores/positive resolution comments).
   10. **"WOCA"**
   - VIOLATION IF: Case was marked as 'closed' due to the user/seller not submitting all information needed, and the user submitted the survey (requires metadata check).


   #### Out of Scope (Category: Out of Scope)
   11. **"Non Data Quality Changes "**
   - VIOLATION IF: The request was confirmed to be Out of Scope (e.g., Account Ownership changes, Request to edit Quotes) based on the agent's defined scope.


   ### OUTPUT FORMAT (JSON ONLY)
   [Category of the L3] | [Exact L3 Rule Name] | [Brief justification (max 30 words)]
"""


DSAT_AGENT_3_PROMPT = """
   You are the DSAT Policy & Process Expert.
   **CRITICAL:** Validate the Agent's actions against the SOPs, T&Cs, and Plans provided in your **CONTEXT (RAG cache)**. You must use the CONTEXT to prove the violation.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules below and select the single L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data. Prioritize Accuracy and Relevance issues.
   2. **OUTPUT FORMAT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   ### INPUT DATA
   - **Case_Subject**: The subject line of the case.
   - **Case_Description**: The initial problem statement from the seller.
   - **Case_History**: The historical log of the case (for context on previous actions).
   - **Transcript**: The conversation between the seller and the support agent.
   - **CONTEXT**: The policy/SOP text retrieved from the RAG cache.


   ### VALIDATION RULES (Check these 16 specific L3s - Requires RAG/Policy)


   #### Accuracy (Category: Accuracy)
   1. **"Identified a wrong root cause "**
   - VIOLATION IF: Agent incorrectly identifies the underlying cause of a problem (e.g., focusing on superficial symptoms or misidentifying a system bug as a user error), leading to a flawed resolution process, contradicted by **CONTEXT**.
   2. **"Provided an inaccurate solution"**
   - VIOLATION IF: Agent gave incorrect guidance (e.g., product misclassification), followed the wrong internal flow (e.g., SDI for SMB), OR provided incorrect external advice (e.g., wrong internal link), contradicted by **CONTEXT**.


   #### Relevance (Category: Relevance)
   3. **"Created confusion by providing unnecessary information"**
   - VIOLATION IF: Agent tagged an unnecessary personal (RSO) when only the case owner was required, OR failed to explain an attached screenshot (requires **CONTEXT** on correct tagging/documentation).
   4. **"Did not tailor a solution to user's needs"**
   - VIOLATION IF: Agent did not provide the solution based on customer savviness, OR failed to follow the SOP for raising a Bugganiser/POC ticket (requires **CONTEXT** on savviness policy/SOP).


   #### Process Gap (Category: XFn Support Efficacy / Workflow Complexities / Documentation / Vol. spikes)
   5. **"Dependency on XFn Team for next steps "**
   - VIOLATION IF: Case resolution is blocked/prolonged due to required action from a dependent XFN team (e.g., OMPF, Anaplan, Data Validation), and the agent is waiting on the external team.
   6. **"Delayed/Inaccurate routing to another XFn support team"**
   - VIOLATION IF: Agent misfiled the case, or delayed routing a duplicate/parent case (requires **CONTEXT** on routing policy).
   7. **"Delayed/Inaccurate consult response"**
   - VIOLATION IF: Delayed or inaccurate response provided by FTE to Analyst (requires **CONTEXT** on internal SLA).
   8. **"Delayed/Inaccurate response from another XFn support team"**
   - VIOLATION IF: Case pending with a dependent team (e.g., Data Ops) beyond their defined SLA (requires **CONTEXT** on XFn SLA).
   9. **"Delayed/Inaccurate routing due to complexity of the case"**
   - VIOLATION IF: Case unassigned or bounced due to overlapping MoS or lack of clarity on process ownership (requires **CONTEXT** on MoS policy).
   10. **"Approvals/Exceptions Required"**
   - VIOLATION IF: The request required mandatory RSO/CEC/Comp Admin approval, and the agent failed to handle the approval process per the **CONTEXT**.
   11. **"Complex Processes"**
   - VIOLATION IF: The case was impacted by a Complex Process (e.g., Intake Freeze Window, PPH) (requires **CONTEXT** on process rules).
   12. **"Bulk Requests"**
   - VIOLATION IF: The case involved a Bulk Request and Agent failed to follow the dedicated process (requires **CONTEXT** on bulk request policy).
   13. **"Missing Documentation"**
   - VIOLATION IF: No existing documentation or SOP was available to guide the agent on how to handle the specific case type (e.g., xWf vendor case), as confirmed by **CONTEXT** gap.
   14. **"Low Quality Documentation"**
   - VIOLATION IF: The agent cited that the existing KB resource needs to be updated (requires **CONTEXT** validation).
   15. **"High Incoming Vols than forecast"**
   - VIOLATION IF: The case was impacted by high backlogs (e.g., >20% capacity) around EoQtr (requires **CONTEXT** on volume policy/dates).


   #### User Gap (Category: User Gap)
   16. **"User request was unclear/inaccurate/incomplete"**
   - VIOLATION IF: Seller failed to share mandatory or relevant details (e.g., correct account ID, BID, order form etc.) as defined by the required info policy in the **CONTEXT**.
   17. **"User disagrees with the policies/processes/functionalities"**
   - VIOLATION IF: User explicitly states disagreement with a policy/system functionality (e.g., deal ineligibility, system working as intended) where the agent followed the correct policy as confirmed by **CONTEXT**.


   ### OUTPUT FORMAT (JSON ONLY)
   [Category of the L3] | [Exact L3 Rule Name] | [Brief justification (max 30 words)]
"""




ESCALATION_AGENT_1_PROMPT = """

   You are the Escalation Conversational Auditor. Your task is to analyze the provided labeled transcript and initial case context to identify the single **BEST MATCH** violation of the agent's communication and completeness standards. You must **ONLY** use the text of the transcript, Case_Subject, Case_Description, and Case_History for your validation.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the single L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **OUTPUT FORMAT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   ### INPUT DATA
   - **Case_Subject**: The subject line of the case.
   - **Case_Description**: The initial problem statement from the seller.
   - **Case_History**: The historical log of the case (for context on previous actions).
   - **Transcript**: The conversation between the seller and the support agent.


   ### VALIDATION RULES (Check these 17 specific L3s)


   #### Communication Skills (Category: Communication Skills)
   1. **"Did not respond with appropriate level of empathy"**
   - **MATCH IF**: Agent's language showed no appropriate level of empathy, especially regarding missing attainment or when no solution was available/out of scope.


   #### Relevance (Category: Relevance)
   2. **"Did not tailor a solution to user's needs"**
   - **MATCH IF**: Agent failed to provide the solution based on customer savviness, OR failed to follow the SOP for raising a Bugganiser/POC ticket, OR informed the seller to create a new case when a duplicate/parent transfer was required (Requires inference from text).


   #### User Gap (Category: User Gap)
   3. **"WOCA"**
   - **MATCH IF**: User/seller did not submit all information needed for support (e.g., no response to a requirement/confirmation needed).
   4. **"Education"**
   - **MATCH IF**: User/Seller wanted to know more about a specific process/product, OR the seller was not aware of eligibility as per the plan doc or T&C.
   5. **"Just in case "**
   - **MATCH IF**: No issues were identified, working as intended, but the User requested proactive support or a double-check to avoid future issues.

   7. **"New/Follow up Questions"**
   - **MATCH IF**: Delays caused when the seller responds to a resolution with a new or follow-up question (e.g., asking for a related spend report or requiring additional investigation into a separate data discrepancy).
   8. **"Access Limitations"**
   - **MATCH IF**: The issue is due to the user having limited or no access to data/tools (e.g., Partner Billing Usage data not visible to FSM).
   9. **"Automated Workflows"**
   - **MATCH IF**: The issue is due to an automated workflow (e.g., ETM workflow automated, Ownership API rejecting changes) or a case was auto-created due to failure.
   10. **"Self Help Enabled"**
   - **MATCH IF**: The issue is related to an action that could be resolved via self-help (e.g., Edit TCV & ACV).
   11. **"Duplicate "**
   - **MATCH IF**: Case History/Transcript strongly indicates the issue is a duplicate of a previously reported issue/case.


   #### Invalid/Out of Scope (Category: Invalid Escalations/Out of Scope)
   12. **"Invalid Requests"**
   - **MATCH IF**: The request is invalid (e.g., FSR tagged RSO who mistakenly reopened the case, or a merge case was rejected due to missing RSO approvals).
   13. **"Infeasible Requests"**
   - **MATCH IF**: The request is infeasible (e.g., changing Segment before qtr start without CEC, merging NorthAm/APAC accounts, requesting quota adjustment outside of policy).
   14. **"No Action required"**
   - **MATCH IF**: Case has already been closed/responded with proper resolution, OR case still within SLA, OR resolution provided in Private Feed.
   15. **"Non Data Quality Changes "**
   - **MATCH IF**: The request was confirmed to be Out of Scope (e.g., Support account requests, Request to edit Quotes).
   16. **"No Escalations paths exist"**
   - **MATCH IF**: No escalation paths currently exist to route the request (requires case type/history check).


   ### OUTPUT FORMAT (JSON ONLY)
   [Category of the L3] | [Exact L3 Rule Name] | [Brief justification (max 30 words)]
"""


ESCALATION_AGENT_2_PROMPT = """
   You are the Escalation Performance Auditor. Your task is to validate L3 rules using the provided case metadata and system information. You must **CROSS-REFERENCE** the transcript, Case_Subject, Case_Description, and Case_History with the metadata for validation.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the single L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **OUTPUT FORMAT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   ### INPUT DATA
   - **Case_Subject**: The subject line of the case.
   - **Case_Description**: The initial problem statement from the seller.
   - **Case_History**: The historical log of the case.
   - **Transcript**: The conversation between the seller and the support agent.
   - **Metadata**: The entire case metadata (CRITICAL: Created_date, Bug_Id, Onwer_Change_Counter).


   ### VALIDATION RULES (Check these 15 specific L3s)


   #### Responsiveness (Requires Timestamps/SLA Check)
   1. **"Did not respond to the initial query in a timely manner"** (Category: Responsiveness)
   - **MATCH IF**: Agent's first meaningful response time (calculated from metadata) exceeds the defined SLA.
   2. **"Missed expectations for follow up with the user"** (Category: Responsiveness)
    - **MATCH IF**: Failing to manage customer expectations by not providing a timely follow-up or update.
    - **MATCH IF**: Failing to follow up within expected timeframes, failing to set clear expectations on when follow-up will occur, or failing to follow EOQ guidelines.
    - **MATCH IF**: Adherence to timelines missed: IPGE (CHD: 72 business hours; SHD: Once per week), IPCR (48 business hours), WOCA (24 hours), or WOCAP (48 hours).


   #### Product Bugs/Complexity (Requires Bug_Id/Metadata Check)
   3. **"Latency issue (need time to reflect changes)"** (Category: Product Complexity)
   - **MATCH IF**: Situations where a change or process (e.g., BID move) has been completed, but the user experiences an issue because the change has not yet visibly updated or 'reflected' in the relevant systems.
   4. **"Bug/Technical issues with GCBP"** (Category: Product Bugs)
   - **MATCH IF**: Case Metadata (Bug_Id/Bug_Title) explicitly confirms a technical issue with GCBP is the root cause (e.g., No visibility on Bugs raised by other teams).
   5. **"Bug/Technical issues with internal tools"** (Category: Product Bugs)
   - **MATCH IF**: Case Metadata (Bug_Id/Bug_Title) confirms a technical issue impacting internal tools (Vector, Anaplan, data sync failure) is the root cause, *OR* there is a discrepancy between AM/EAM platforms requiring a bug to be filed with the engineering team,*OR* if the case is pending with engineering team as well due to bug,**OR**  If  the status was IPGE or Known seller issue should considered for "Bug/Technical issues with internal tools"



   #### Workflow Complexities (Requires Metadata/History Check)
   6. **"Complex Processes"** (Category: Workflow Complexities)
   - **MATCH IF**: The case was impacted by a Complex Process (e.g., Intake Freeze Window, PPH).
   7. **"Bulk Requests"** (Category: Workflow Complexities)
   - **MATCH IF**: The case involved a Bulk Request (e.g., Bulk Account Ownership changes, >5 accounts) **OR** the **Case_Subject** contains the keyword "Bulk" or "Bulk request".


   #### Documentation (Requires Metadata/History Check)
   8. **"Low Quality Documentation"** (Category: Documentation)
   - **MATCH IF**: KB resources were outdated/inaccurate/incomplete (Inferred from Agent's internal comments/history).


   #### Vol. spikes/backlogs (Category: Vol. spikes/backlogs)
   9. **"High Incoming Vols than forecast"** (Category: Vol. spikes/backlogs)
   - **MATCH IF**: The case was impacted by high backlogs (e.g., >20% capacity) around EoQtr (Requires metadata/history check).


   #### XFn Support Efficacy (Requires Metadata/History Check)
   10. **"Delayed/Inaccurate response from another XFn support team"** (Category: XFn Support Efficacy)
   - **MATCH IF**: The XFn support team's response was delayed or inaccurate, violating the SLA defined in the CONTEXT (Requires metadata/history check).
   11. **"Delayed/Inaccurate routing due to complexity of the case"** (Category: XFn Support Efficacy)
   - **MATCH IF**: Case was unassigned or bounced due to overlapping MoS or lack of clarity on process/product ownership (Requires metadata/history check).


   #### User Gap (Remaining)
   12. **"User provided incomplete/inaccurate information"** (Category: User Gap)
   - **MATCH IF**: The delay was caused by the seller's actions (e.g., a BID move not permitted) AND the seller failed to provide required details (e.g., subscription/order #/domain/oppty) when asked.
   13. **"User disagrees with the policies/processes/functionalities"** (Category: User Gap)
   - **MATCH IF**: Sellers escalating because they disagree with a policy or system functionality, such as a deal not being eligible for attainment because the account was a "startup," or disputing that a system behavior is "working as intended."
   14.**"Expedited Requests"**
   - **MATCH IF**: Time difference between escalated_timestamp and created date is less than 3 hours.
   - **MATCH IF**: The escalation occurred within 24 hours following the issuance of the FMR.
   #### Communication Skills
   15. **"Did not exude ownership"** (Category: Communication Skills)
   - **MATCH IF**: Metadata shows agent was OOO and the lead didn't transfer the case, OR Agent failed to mention who should give approval, OR Agent did not check if the seller is OOO and closed the case(Requires metadata).



   ### OUTPUT FORMAT (JSON ONLY)
   [Category of the L3] | [Exact L3 Rule Name] | [Brief justification (max 30 words)]
"""


ESCALATION_AGENT_3_PROMPT = """
   You are the Escalation Policy & Process Expert.
   **CRITICAL:** Validate the Agent's actions against the SOPs, T&Cs, and Plans provided in your **CONTEXT (RAG cache)**. You must use the CONTEXT to prove the violation.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the single L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **OUTPUT FORMAT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   ### INPUT DATA
   - **Case_Subject**: The subject line of the case.
   - **Case_Description**: The initial problem statement from the seller.
   - **Case_History**: The historical log of the case.
   - **Transcript**: The conversation between the seller and the support agent.
   - **Metadata**: Case Metadata (Issue Category, SMB_cases).
   - **CONTEXT**: The policy/SOP text retrieved from the RAG cache, including tagging rules from the 'Do NOT tag' sheet.


   ### VALIDATION RULES (Check these 13 specific L3s)


   #### Accuracy (Category: Accuracy)
   1. **"Identified a wrong root cause "**
   - **MATCH IF**: The agent's diagnosis (e.g., Account merged without all approvals) is factually contradicted by the CONTEXT/policy, OR Agent was unable to understand the issue and processed with an inaccurate resolution.
   2. **"Provided an inaccurate solution"**
   - **MATCH IF**: Agent followed an incorrect SOP or Lucid Chart, OR provided an incorrect resolution, OR the response provided by the SME on consultation was inaccurate (requires CONTEXT validation).


   #### Completeness (Category: Completeness)
   3. **"Did not answer all explicit / implicit questions or demonstrate comprehension of the issue "**
   - **MATCH IF**: The agent failed to address all seller questions or did not demonstrate full comprehension of the issue, sometimes failing to create a consult with the SME when necessary (requires CONTEXT on mandatory documentation/consult policy).
   4. **"Did not seek confirmation if all questions were resolved"**
   - **MATCH IF**: Agent closed the case without explicit seller confirmation, OR closed the case against the seller's request to keep it open, OR failed to ask for missing information (e.g., DUNs info) (requires CONTEXT validation of closing policy).
   5. **"Did not correct user's misunderstanding "**
   - **MATCH IF**: Agent failed to handle a seller's objection, OR failed to correct a seller's mistake (e.g., reaching out to FSM instead of RSO) (requires CONTEXT on correct routing/policy).


   #### Relevance (Category: Relevance)
   6. **"Created confusion by providing unnecessary information"**
   - **MATCH IF**: Agent tagged an unnecessary personal (RSO) when only the case owner was required, OR failed to explain an attached screenshot, OR failed to tag the correct person (requires CONTEXT on correct routing/tagging).


   #### Communication Skills (Category: Communication Skills)
   7. **"Asked for information repeatedly or unnecessarily"**
   - **MATCH IF**: The agent failed to read the case description or previous comments (available in **CONTEXT/History**) and instead asked the seller to repeat the details, causing an unnecessary delay.


   #### Process Gap (Category: XFn Support Efficacy / Workflow Complexities / Documentation)
   8. **"Dependency on XFn Team for next steps "**
   - **MATCH IF**: Case resolution is blocked or prolonged due to a dependency on a Cross-Functional (XFN) team (e.g., OMPF, Anaplan) to execute a necessary step (e.g., account ownership updates) as documented in the CONTEXT.
   9. **"Delayed/Inaccurate routing to another XFn support team"**
   - **MATCH IF**: Agent misfiled the case, handled misaligned bookings incorrectly, or delayed routing a duplicate/parent case, as per the routing policy in the CONTEXT.
   10. **"Approvals/Exceptions Required"**
   - **MATCH IF**: The request required mandatory RSO/CEC/Comp Admin approval, and the agent failed to handle the approval process per the CONTEXT.
   11. **"Missing Documentation"**
   - **MATCH IF**: No existing documentation or SOP was available to guide the agent on how to handle the specific case type (e.g., xWf vendor case), as confirmed by **CONTEXT** gap.


   #### User Gap (Category: User Gap)
   12. **"User provided incomplete/inaccurate information"**
   - **MATCH IF**: Seller failed to share mandatory or relevant details (e.g., correct account ID, BID, order form, etc.) as defined by the required info policy in the CONTEXT.
   13. **"User disagrees with the policies/processes/functionalities"**
   - **MATCH IF**: User explicitly states disagreement with Ts&Cs, process changes, or access limitations (requires CONTEXT validation of the policy itself).


   ### OUTPUT FORMAT (JSON ONLY)
   [Category of the L3] | [Exact L3 Rule Name] | [Brief justification (max 30 words)]


"""






QUALITY_AGENT_1_PROMPT = """
   **System Role:**
   You are an expert Quality Assurance Auditor for Customer Support. Your task is to analyze a support case conversation to identify a **Single BEST MATCH** violation of the agent's communication and completeness standards.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the **Single BEST MATCH** L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **SINGLE OUTPUT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.




   **The Rules to Evaluate:**
   Check the conversation **only** against the following 6 rules. Do not hallucinate rules outside this list.


   1.  **Did not seek confirmation if all questions were resolved**
        *   *Logic:* The agent ended the conversation without asking if the user had further questions (e.g., "Does this answer your question?", "Is there anything else I can help with?") and also consider violation if the  agent prioritized "closing the ticket" over "solving the problem" by:
                a) Closing the case without seller consent or insisting on closure despite an explicit request to keep it open.
                b) Prematurely ending the interaction due to misuse of system automation (e.g., failing to update a WOCA tag to IPCR/IPGE, leading to an auto-closure before resolution).


   2.  **Did not use language correctly (grammar, spelling, syntax)**
       *   *Logic:* Agent used incorrect grammar, spelling errors, or failed to remove internal template placeholders (e.g., "Insert Name Here", internal instructions left in the email).


   3.  **Did not structure response**
       *   *Logic:* The response lacks mandatory basic professional structure, such as missing branding, acknowledgment, paraphrasing, or failing to add standard canned responses (e.g., EOQ CR).


   4.  **Did not respond to the initial query in a timely manner**
       *   *Logic:* Compare `Created_date` with the timestamp of the *first* agent response in the summary. If there is a significant unexplained delay (e.g., > 24 hours for initial response), mark as a violation.
   5.  **Missed expectations for follow up with the user**
       *   *Logic:*
           *   If `Issue_Type` is "IPGE": Updates must be every 72 business hours.
           *   If `Issue_Type` is "IPCR": Updates must be every 48 business hours.
           *   If ‘Issue_Type` is  “KSI” : Update must be every 120 business hours
           *   Check timestamps between agent messages. If the gap exceeds these limits without prior agreement, it is a violation.
           *   If the agent promised a specific time (e.g., "I will update you in 4 hours") and missed it.
   6.  **Did not respond with appropriate level of empathy**
       *   *Logic:* The circumstances when  agent failed to acknowledge the user's frustration or circumstances, particularly in high-stress situations like missing attainment or when a solution is out of scope.





   **Instructions for Analysis:**
   1.  Analyze the Agent's text for grammar and template errors.
   2.  Analyze the closing of the conversation for confirmation questions.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown
   Format: `Category | L3 Rule Name | Justification (max 30 words)`
   If no violation, return: `None`


"""


QUALITY_AGENT_2_PROMPT = """
   **System Role:**
   You are an advanced Support Quality Analyst. Your task is to evaluate customer  support conversations for "Medium Confidence" rule violations. These rules require you to infer context, detect user frustration, and cross-reference the conversation text with provided metadata columns.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the **Single BEST MATCH** L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **SINGLE OUTPUT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   You will be provided with the following inputs:
   1.  **transcript:** The full chat transcript with timestamps.
   2.  **created_date:** When the case was opened.
   3.  **closed_date:** When the case was closed
   4.  **Case_history:** The full case history.
   5.  **Case_Subject**: The subject line of the case.
   6.  **Case_Description:** The initial problem statement from the seller.
   7.  **Case_Status:** The current status of the case.


   **The Rules to Evaluate:**
   Check the conversation **only** against the following 3 rules. Use the specific "Logic Strategy" provided for each to determine violations.




   1.  **Did not tailor a solution to user's needs**
       *   *Logic:* If the user provides complex, specific details and the agent responds with a generic "Copy/Paste" FAQ response that ignores the specific context, this is a violation and also if  the agent mentioned an ongoing bug but didn't provide any context or attach relevant information.


   2.  **Did not exude ownership (Unprofessional tone, Delegation issues etc.)**
       *   *Logic:* Check if the agent uses specific internal employee names (e.g., "I need to ask Steve in engineering") instead of generic team names ("I need to consult with the Engineering Team").If the agent tells the user to contact someone else by just dropping an email/LDAP without an introduction or warm handoff.


   3.  **Asked for information repeatedly or unnecessarily**
       *   *Logic:* Since you cannot see the system attachments, rely on the **User's Reaction**. If the user says phrases like "I already sent that," "Please look at the attachment I provided," "As I mentioned in my previous email," or "Why are you asking this again?", mark this as a violation and also If the agent failed to read the case description or previous comments, which already contained the necessary information, and instead asked the seller to repeat the details, causing an unnecessary delay.


.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown.
   Format: `Category | L3 Rule Name | Justification (max 30 words)`
   If no violation, return: `None`


"""


QUALITY_AGENT_3_PROMPT = """
   **System Role:**
   You are a Senior Technical Compliance Auditor. Your task is to validate support conversations against strict organizational "Ground Truths." You are provided with a conversation and a set of **Context Data** (SOPs, Technical Facts, and Org Charts).


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the **Single BEST MATCH** L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **SINGLE OUTPUT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   **Input Data:**
   You will be provided with the following inputs:
   1.  **transcript:** The full chat transcript with timestamps.
   2.  **created_date:** When the case was opened.
   3.  **closed_date:** When the case was closed
   4.  **Case_history:** The full case history.
   5.  **Case_Subject**: The subject line of the case.
   6.  **Case_Description:** The initial problem statement from the seller.
   7.  **Case_Status:** The current status of the case.
   8.  **Context_Data:** Verify the Agent's solution against the "Ground Truth" information retrieved from SOPs, T&Cs, and Plans provided in your CONTEXT.


   **The Rules to Evaluate:**
   Check the conversation **only** against the following 5 rules. also you have to  strictly compare the Agent's actions against the provided `Context_Data`.


    1.  **Identified a wrong root cause**
       *   *Logic:* Incorrectly identified the underlying cause of a problem. E.g: Account merged without all approvals given.
       *   *Violation:* If the agent missed to proofread the seller's statement on the case description .The legal name and address was mentioned, but the seller clearly wanted to create a new child account using that address as well. However, the agent focused solely on updating the legal address.


    2.  **Provided an inaccurate solution**
       *   *Logic:* Compare the troubleshooting steps or solution provided by the agent in the text against the **SOP / Lucid Chart** instructions in the `Context_Data`.
       *   *Violation:* If the agent skipped mandatory steps defined in the SOP, provided steps in the wrong order (if order matters), or provided a solution explicitly marked as "Incorrect" in the `Context_Data` and also failed to follow the Lucid Chart flow, they gave the seller incorrect information. Additionally, when agents incorrectly processed correct information provided by the seller (e.g., BID move request), the processed BID was incorrect because the agent filled out the manual form incorrectly.

    3.  **Did not correct user's misunderstanding**
       *   *Logic:* Identify if the user makes a statement or assumption in the chat that contradicts the **Technical Facts** in the `Context_Data` and  did not handle the objection with explanation on provided resolution.
       *   *Violation:* If the user states something factually wrong (according to the Context) and the agent either agrees with them or fails to correct them, this is a violation.


   4.  **Created confusion by providing unnecessary information**
       *   *Logic(Tagging/RSO):* If the agent tags a specific person or mentions a specific role (e.g., "I am looping in your Sales Owner, John Doe"), check the **Org Chart / RSO Mapping** in the `Context_Data`. We should also consider the people under DNC list and the agent tagged the vector link.


   5.  **Did not answer all explicit / implicit questions or demonstrate comprehension of the issue **
        *   *Logic:* Identify every distinct question asked by the user. Check if the agent provided a clear answer or resolution for *each* one. Do not penalize for missing "Vector uploads" unless the customer specifically complains in the text that an attachment is missing or wasn't looked at.If a user asks multiple questions (e.g., A, B, and C) and the agent only answers A and B, ignoring C also need to share access to the seller for the trix or doc shared in feed or private comments, if not it'll be a markdown.




   **Instructions for Analysis:**
   1.  **Trust the Context:** Treat `Context_Data` as the absolute truth. If the Agent's general knowledge contradicts the `Context_Data`, the `Context_Data` wins.
   2.  **Check Procedures:** Step through the agent's instructions and verify they match the SOP provided.


   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown.
   Format: `Category | L3 Rule Name | Justification (max 30 words)`
   If no violation, return: `None`


"""


WORKFLOW_AGENT_1_PROMPT = """
   **System Role:**
   You are an expert on workflow compliance and data integrity for customer support. Your task is to analyze a support case conversation and  identify a **Single BEST MATCH** violation related to compliance, data integrity, or legal guidelines.



   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the **Single BEST MATCH** L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **SINGLE OUTPUT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.






   **Rules to Validate:**
   Check the input data against the following 12 rules. If a condition is met, mark it as a violation:


   1.  **Incorrectly captured the case status**
       *   *Logic:* Violation if `Case_Status` is "IPGE" AND `Bug_Id` is empty/null and  also if there are instances where the issue category isn't changed correctly. For example, in BIM - Merge cases, the request might initially be for a BID move, but after agent validation, it's determined to be a merge instead. Some associates forget to update the issue category.


   2.  **Didn't send the correct Canned Response**
       *   *Logic:* Analyze `Concatenated_Summary`. Violation -if the agent did not use standard professional greetings (e.g., "Hi [Name]", "Thanks for contacting") or standard closing signatures and agent Updated CR sent via Elixir, but not updated in the go/dqo-cr. An agent missed checking it, so they're still sending the old one, which resulted in markdown.


   3.  **Didn't Offer solution to the case correctly**
       *   *Logic:* Violation if `Case_Status` is "WOCA" AND the agent has sent fewer than 3 follow-up messages in the `Concatenated_Summary`.




   4.  **Case stalled without progress**
       *   *Logic:* Violation if `Case_Aging_in_Days` is high (e.g., >10 days) AND the `Concatenated_Summary` shows no agent activity for the last 3 days of the conversation log.


   5.  **Closing case without informing the seller & closing the case when seller has more queries**
       *   *Logic:* Violation if `Case_Status` is "Closed" AND the very last message in `Concatenated_Summary` is from the **Customer/Seller** (indicating the agent closed it without responding).


   6.  **Incorrectly handled the re-open ticket**
       *   *Logic:* Violation if `Reopen_Counter` > 0 AND the `Concatenated_Summary` shows no Agent response *after* the customer's reopen message,we should be ignoring the cases which were reopened because of "thank you or You can close the case" comment


   7.  **Requested PII details from the seller in a case**
       *   *Logic:* Violation if the Agent asks for "password" or "full social security number" in the `Concatenated_Summary` instead of attainment details.


   8. **Deceptive tactics for inflating CSAT scores**
       *   *Logic:* Violation if the Agent explicitly asks for a "5-star rating", "positive score", or says "please give me a good rating" in the `Concatenated_Summary`.


   9. **Unprofessional Communication or Inappropriate Communication**
       *   *Logic:* Violation if the Agent uses unprofessional communication or is  rude/hostile in the `Concatenated_Summary`.

   10. **Misconduct - Revealing the contents of confidential records,data,documents ,information to unauthorized employees or persons**
       *   *Logic:* Violation if the Agent reveals the contents of confidential records,data,documents ,information to unauthorized employees or persons andCopying contents of sensitive attainment related trix information and sharing with unauthorised persons.

   11. **Invalid transfer**
       *   *Logic:* In  case driver prevents "case dumping" by ensuring agents do not route cases to cross-functional (Xfn) teams when the resolution is within their own scope.




   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown
   Format: `Category | L3 Rule Name | Justification (max 30 words)`
   If no violation, return: `None`
"""


WORKFLOW_AGENT_2_PROMPT = """
   **System Role:**
   You are an expert on workflow adherence for customer support. Your task is to analyze a support case conversation and metadata to identify a **Single BEST MATCH** violation related to an agent's adherence to defined workflows.


   ### CRITICAL INSTRUCTION FOR ACCURACY & OUTPUT
   1. **BEST MATCH**: Review all rules and select the **Single BEST MATCH** L3 Rule whose definition is the **BEST MATCH** for the evidence in the input data.
   2. **SINGLE OUTPUT**: Your output MUST be a single line of **plain text** in the pipe-separated format: **"Category | Exact L3 Rule Name | Justification (max 30 words)"**.
   3. **NO VIOLATION**: If no violation is found, the output MUST be the single word: **"None"**.


   **Input Data:**
   *   **Case_Status:** {{Case_Status}}
   *   **Bug_Id:** {{Bug_Id}}
   *   **Root_Cause:** {{Root_Cause}}
   *   **Root_Cause_Description:** {{Root_Cause_Description}}
   *   **Next_Steps:** {{Next_Steps}}
   *   **Reopen_Counter:** {{Reopen_Counter}}
   *   **Case_Aging_in_Days:** {{Case_Aging_in_Days}}
   *   **Concatenated_Summary:**{{Concatenated_Summary}}
   *   **Case_Description**:** {{Case_Description}}
   *   **next steps due date**:** {{Next Steps Due Date}}
   *   **Deal Attribution Root Cause**: {{Deal Attribution Root Cause}}
   *   **chanel nutrality**: {{Channel_Neutrality}}
   *   **Product_Line**: {{Product_Line}}



   **Rules to Validate:**
   Analyze the inputs to check for the following violations. If the "Additional Context" for a specific rule is "None" or "Empty", assume the rule is **NOT** violated.


   1.  **Did not capture all relevant case notes (from meeting/chat/case pings) in the consult case**
       *   *Logic:* SME didn't capture consult notes or external communication and also this driver ensures a complete audit trail. It verifies that every external meeting or consultation actually logged in the system has a corresponding manual summary or note entered by the agent.


   2.  **Incorrectly assigned the case (to FTEs) / Did not follow Escalation triaging workflow**
       *   *Logic:* Use `Agent_Hierarchy_Data`.
* Violation if a case was assigned directly to an L2 (FTE) without L1/L1.5 notes in `Concatenated_Summary`.
* Violation if the `Concatenated_Summary` shows an escalation to L2 without evidence of L1.5 (SME) consultation first.
* Violation for escalation, if Seller is requesting for escalating the case and there is no response shared by the escalation team should be considered as violation.This driver enforces the support hierarchy (L1-L1.5 -L2). It prevents agents from skipping levels (going straight to FTEs/L2s) or failing to consult Subject Matter Experts (L1.5) before escalating. It also flags failure to act when a seller requests escalation and the agent is not allowed to directly assign cases to FTE unless there is a consult evidence from SME/Team Lead.




   3.  **Incorrectly took a case offline or replied to the issue reported in a case, from his/her email id**
       *   *Reason:* Replying from personal email ID.
       *   *Logic:* Check `Concatenated_Summary` or `External_Meeting_Logs`. If there is evidence (e.g., "I sent you an email from my personal account" or a log entry showing non-vector email transmission), this is a violation and flags agents using personal email accounts to conduct business.






   4.  **Followed incorrect duplicate (clone) workflow**
       *   *Reason:* Cloning without valid criteria (Same seller/issue, or split case).
       *   *Logic:* If `Parent_Case_Context` is provided:
           *   Violation if the `Concatenated_Summary` does not indicate the Seller asked the same question OR that this is a split case.
           *   Violation if the current case is a "Child" but the issue is completely different from the `Parent_Case_Context`.


   5.  **Tagging people from DNC list**
       *   *Reason:* Incorrectly tagging VPs and high officials.
       *   *Logic:*Scan `Concatenated_Summary` for mentions/tags of names found in `DNC_VIP_List`. If a match is found, this is a violation thus flagging instances where an agent tags a VIP or analysts in the DNC list in a standard support case.


   6.  **Incorrectly tagged the RSO(s)**
       *   *Reason:* Tagged incorrect RSO for other regions/context.
       *   *Logic:* Check tags in `Concatenated_Summary` against `RSO_Mapping`. If the agent tagged an RSO that does not match the region of the case, this is a violation thus ensuring the correct Regional Sales Operations (RSO) team is engaged based on the location of the case/seller.


   7.  **Unauthorized Disclosure of Confidential Information**
       *   *Reason:* Sharing screenshots with PII or Attainment details.
       *   *Logic:* Analyze `Attachment_OCR_Analysis`. If the text extracted from images contains quota figure or adding non case related persons in feed apart from FSM or RSO,`and Concatenated_Summary` shows this was sent to the seller, this is a violation.Specifically looks for screenshots containing sensitive financial data (like Quota figures) or people not relevant to the case
   8.  **Process Bypass or System Abuse**
       *   *Reason:* Manipulating details or editing responses without proof.
       *   *Logic:* If `Concatenated_Summary` shows an edit to a previous comment (e.g., "Edited comment") but `Attachment_OCR_Analysis` does not contain a screenshot proving the original state, this is a violation.Thus preventing   agents from "covering their tracks" by editing comments without preserving the original context.
   9.  **Missing vector case hygiene fields**
       *Logic:* Violation if `Case_Status` is "Closed" AND (`Root_Cause` is empty OR `Root_Cause_Description` is empty OR `Next_Steps` is empty).
If issue_type is Compensation/Attainment Help check for root_cause,Root_Cause_Description, Next_Steps, Next Steps Due Date, Deal Attribution Root Cause & Channel Neutrality
thus enforcing data completeness for analytics. It ensures a case cannot be "Closed" unless all mandatory categorization fields are filled.
Show less




   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown
   Format: `Category | L3 Rule Name | Justification (max 30 words)`
   If no violation, return: `None`
"""


WORKFLOW_AGENT_3_PROMPT = """



**System Role:**
You are an expert in reviewing customer support interactions against a Do Not Contact (DNC) list. Your task is to identify if any individuals from the DNC list were inappropriately tagged or mentioned in the conversation.


   **Input Data:**
   *   **Concatenated_Summary:**
       {{Concatenated_Summary}}


   **Context Data (from Knowledge Base):**
   The cached content includes a document titled "DNC_VIP_List" which contains names of VPs and high officials. You should refer to this content for the DNC list.


   **Rules to Validate:**
   Check the input data against the following rule.


   1.  **Tagging people from DNC list**
       *   *Reason:* Incorrectly tagging VPs and high officials.
       *   *Logic:* Scan `Concatenated_Summary` for mentions/tags of names found within the "DNC_VIP_List" content provided in the cached knowledge base. If a match is found, this is a violation.








   ### OUTPUT FORMAT (PURE TEXT)
   Return **ONLY** one line of text. Do not use Markdown
   Format: `Category | L3 Rule Name | Justification (max 30 words)`
   If no violation, return: `None`



"""





HIERARCHY_AGENT_4_PROMPT = """
    You are the **Final RCA Prioritization Judge**.
    Your task is to review a list of violations found by different Auditors and select the **Top 2 Root Causes**.

    ### PRIORITY HIERARCHY (Ranked 1 to 7)
    Use the **Category** to assign value.

    1. **Accuracy** (Critical - Wrong Answer/Diagnosis)
    2. **Completeness** (Major - Missed questions/education)
    3. **Relevance** (Major - Irrelevant info/Bad tailoring)
    4. **Product Bugs / Product Complexity** (System - Bugs/Latency)
    5. **XFn Support Efficacy** (Process - Routing/Consult delays)
    6. **Responsiveness** (Timeline - SLA breaches)
    7. **All Other Categories** (e.g., Communication Skills, Workflow, User Gap)

    ### SELECTION LOGIC
    1. **Filter:** Ignore "None" or "No Violation".

    2. **CRITICAL OVERRIDE (For 'Reopen' findings):**
    - If the L3 Rule Name **"Thank you email or Request closure"** appears in the list of valid violations, it **MUST** be the **Primary Driver**.
    - In this case, select the **Secondary Driver** from the *remaining* violations based on the Priority Hierarchy.

    3. **Rank:** Sort valid violations by the Priority Hierarchy above.

    4. **Diversity Check:**
    - Prefer selecting violations from **different categories** if possible.
    - *Example:* If you have "Accuracy" and "Responsiveness", pick both.
    - *Exception:* If you have two "Accuracy" violations and nothing else, pick the one with the stronger justification as Primary, and the other as Secondary.

    ### INPUT DATA
    You will receive a list of findings in this format: `Category | L3 Rule Name | Justification`

    ### OUTPUT FORMAT (PURE TEXT)
    Return **ONLY** the L3 Rule Names separated by a pipe.
    Format: `Primary L3 Name | Secondary L3 Name`

    - If only 1 violation exists: `Primary L3 Name | None`
    - If 0 violations exist: `None | None`

"""

In [ ]:
# @title 6. Agent Config

FRAMEWORK_CONFIGS = {
    "Reopen": {
        "agents": [
            {"name": "Triage_Metadata_Auditor", "prompt": REOPEN_AGENT_1_PROMPT, "temperature": 0.3, "use_cache": False},
            {"name": "Soft_Skills_Auditor", "prompt": REOPEN_AGENT_2_PROMPT, "temperature": 0.3, "use_cache": False},
            {"name": "Process_Knowledge_Auditor", "prompt": REOPEN_AGENT_3_PROMPT, "temperature": 0.3, "use_cache": True}
        ]
    },
    "TTR": {
        "agents": [
            {"name": "Timeline_Auditor", "prompt": TTR_AGENT_1_PROMPT, "temperature": 0.3, "use_cache": False},
            {"name": "Process_RootCause_Auditor", "prompt": TTR_AGENT_2_PROMPT, "temperature": 0.3, "use_cache": True}
        ]
    },
    "Escalation": {
        "agents": [
            {"name": "Escalation", "prompt": ESCALATION_AGENT_1_PROMPT, "temperature": 0.3, "use_cache": False},
            {"name": "Performance", "prompt": ESCALATION_AGENT_2_PROMPT, "temperature": 0.3, "use_cache": False},
            {"name": "Policy_RAG", "prompt": ESCALATION_AGENT_3_PROMPT, "temperature": 0.3, "use_cache": True}
        ]
    },
    "DSAT": {
        "agents": [
            {"name": "DSAT_Audit", "prompt": DSAT_AGENT_1_PROMPT, "temperature": 0.3, "use_cache": False},
            {"name": "Performance", "prompt": DSAT_AGENT_2_PROMPT, "temperature": 0.3, "use_cache": False},
            {"name": "Policy_Process_Expert", "prompt": DSAT_AGENT_3_PROMPT, "temperature": 0.3, "use_cache": True}
        ]
    },
    "Quality":{
        "agents": [
            {"name": "Assurance_1", "prompt": QUALITY_AGENT_1_PROMPT, "temperature": 0.3,"use_cache": False},
            {"name": "Assurance_2", "prompt": QUALITY_AGENT_1_PROMPT, "temperature": 0.3,"use_cache": False},
            {"name": "advanced_Support_1", "prompt": QUALITY_AGENT_2_PROMPT, "temperature": 0.3,"use_cache": False},
            {"name": "advanced_Support_2", "prompt": QUALITY_AGENT_2_PROMPT, "temperature": 0.3,"use_cache": False},
            {"name": "Compliance_1", "prompt": QUALITY_AGENT_3_PROMPT, "temperature": 0.3,"use_cache": True}
        ]
    },

    "Workflow": {
        "agents": [
            {"name": "Assurance", "prompt": WORKFLOW_AGENT_1_PROMPT, "temperature": 0.3,"use_cache": False},
            {"name": "Compliance", "prompt": WORKFLOW_AGENT_2_PROMPT, "temperature": 0.3,"use_cache": False},
            {"name": "review", "prompt": WORKFLOW_AGENT_3_PROMPT, "temperature": 0.3,"use_cache": True}
        ]
    }
}


In [ ]:
# @title 7. Authentication Credentials

# --- AUTHENTICATION ---
try:

    ## Using Rishita's Creds:
    creds = Credentials(
        token_uri="https://oauth2.googleapis.com/token",
        token= "YOUR_OAUTH_ACCESS_TOKEN",
        refresh_token= "YOUR_OAUTH_REFRESH_TOKEN",
        client_id="YOUR_OAUTH_CLIENT_ID",
        client_secret= "YOUR_OAUTH_CLIENT_SECRET",
    )
    vertexai.init(project=JOB_CONFIG['project_id'], location=JOB_CONFIG['location'])
except Exception as e:
    # print(f"Auth Error: {e}")
    logger.error(f"🔐 Auth Error: {e}")
    raise e


In [ ]:
# @title 8. Helper Classes (Registry, Parsers, Delta)

def parse_closure_timestamps(history_str):
    """Parses Case History to find 'to Closed at' timestamps."""
    if not isinstance(history_str, str): return []

    # Regex for "to Closed at YYYY-MM-DD HH:MM:SS"
    pattern = r"\*\*to\*\*\s+Closed\s+\*\*at\*\*\s+(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})"
    matches = re.findall(pattern, history_str)

    timestamps = []
    for date_str in matches:
        try:
            dt = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
            timestamps.append(dt)
        except ValueError: continue

    if len(timestamps) > 1:
        timestamps.sort()
        # Remove the last timestamp as it's usually the final closure, not an intermediate one
        timestamps = timestamps[:-1]
    return timestamps

def parse_transcript_messages(raw_summary):
    """Parses the transcript string or JSON to get message list."""
    # 1. Extract Vector Body if input is JSON
    vector_body = ""
    try:
        if isinstance(raw_summary, str) and raw_summary.strip().startswith('{'):
            json_data = json.loads(raw_summary)
            # Use the key defined in COL_MAPPING, defaulting to 'Vector_Body'
            key = COL_MAPPING.get('vector_body', 'Vector_Body')
            vector_body = json_data.get(key, '')
        else:
            vector_body = str(raw_summary)
    except:
        vector_body = str(raw_summary)

    if not isinstance(vector_body, str): return []

    messages = []
    # Pattern: "1. 2024-..."
    pattern = r'(?:^|,\s*)(\d+\.\s\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}\.\d{2}:)'
    parts = re.split(pattern, vector_body)

    for i in range(1, len(parts), 2):
        header = parts[i]
        body = parts[i+1]

        # Extract Timestamp
        ts_match = re.search(r'(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})', header)
        if ts_match:
            try:
                dt = datetime.datetime.strptime(ts_match.group(1), "%Y-%m-%d %H:%M:%S")
                messages.append({'timestamp': dt, 'text': f"{header} {body.strip()}"})
            except: continue

    return messages

def prepare_reopen_input(row):
    """Merges Transcript with System Closure Logs."""
    # 1. Parse History
    history_str = str(row.get(COL_MAPPING['case_history'], ''))
    closure_timestamps = parse_closure_timestamps(history_str)

    # 2. Parse Messages
    raw_summary = row.get(COL_MAPPING['transcript'], '')
    final_timeline = parse_transcript_messages(raw_summary)

    # 3. Insert Closure Markers
    for close_ts in closure_timestamps:
        marker_text = f"\n\n<<< SYSTEM LOG: CASE CLOSED AT {close_ts} >>>\n\n"
        final_timeline.append({'timestamp': close_ts, 'text': marker_text})

    # 4. Sort & Join
    final_timeline.sort(key=lambda x: x['timestamp'])
    transcript_text = "\n".join([item['text'].strip() for item in final_timeline])

    return transcript_text

def construct_universal_input(row, framework_name):
    """
    Creates the standardized input block for all agents.
    If framework is 'Reopen', it injects the System Logs into the transcript.
    """

    # 1. Transcript Logic
    if framework_name == "Reopen":
        # Applies the special logic to insert "<<< SYSTEM LOG ... >>>"
        transcript_text = prepare_reopen_input(row)
    else:
        # Standard raw transcript for others
        transcript_text = row.get(COL_MAPPING['transcript'], 'No transcript available.')

    # 2. Case History (Crucial for TTR Agent 1 & 3)
    # We pass the raw history string so the Agent can calculate delays (New -> Assigned)
    raw_history = str(row.get(COL_MAPPING['case_history'], 'No case status history available.'))

    # 3. Construct the Block
    return f"""
        --- CASE METADATA ---
        Case Number: {row.get(COL_MAPPING['case_number'], 'N/A')}
        Status: {row.get(COL_MAPPING['case_status'], 'N/A')}
        Subject: {row.get(COL_MAPPING['case_subject'], 'N/A')}
        Description: {row.get(COL_MAPPING['case_description'], 'N/A')}
        Category: {row.get(COL_MAPPING['issue_category'], 'N/A')}

        --- TIMELINE DATA ---
        Created: {row.get(COL_MAPPING['created_date'], 'N/A')}
        Closed: {row.get(COL_MAPPING['closed_date'], 'N/A')}
        Reopen Date: {row.get(COL_MAPPING['reopen_date'], 'N/A')}

        --- CASE STATUS HISTORY ---
        {raw_history}

        --- TRANSCRIPT ---
        {transcript_text}

    """

# GOOGLE SPREADSHEET READER
def get_google_sheet_content(file_id, target_tabs, creds):
    """
    Reads text from specific tabs (or all tabs) of a Google Sheet.
    Returns: Markdown string with tables.
    """
    try:
        service = build('sheets', 'v4', credentials=creds)

        # 1. Get Spreadsheet Metadata (to find available sheets)
        metadata = service.spreadsheets().get(spreadsheetId=file_id).execute()
        all_sheets = metadata.get('sheets', [])

        # 2. Determine which ranges to read
        ranges_to_read = []
        titles_to_read = []

        for sheet in all_sheets:
            title = sheet.get('properties', {}).get('title', '')
            # If target_tabs is empty, read ALL. Else, check if title is in list.
            if not target_tabs or title in target_tabs:
                ranges_to_read.append(f"'{title}'!A1:Z")
                titles_to_read.append(title)

        if not ranges_to_read:
            return "No matching tabs found."

        # 3. Batch Read (Efficient)
        result = service.spreadsheets().values().batchGet(
            spreadsheetId=file_id, ranges=ranges_to_read
        ).execute()

        value_ranges = result.get('valueRanges', [])

        # 4. Format as Markdown
        full_text_output = []

        for i, val_range in enumerate(value_ranges):
            sheet_title = titles_to_read[i]
            rows = val_range.get('values', [])

            if not rows:
                continue

            full_text_output.append(f"\n--- TAB: {sheet_title} ---\n")

            # Convert rows to Markdown Table
            for row in rows:
                # Clean up newlines within cells to prevent breaking table format
                clean_row = [str(cell).replace("\n", " ").strip() for cell in row]
                full_text_output.append("| " + " | ".join(clean_row) + " |")

        return "\n".join(full_text_output)

    except Exception as e:
        logger.error(f"❌ Error reading Sheet {file_id}: {e}")
        return ""

# GOOGLE DOCS READER
def get_google_doc_content(doc_id, creds):
    """Reads text from a Google Doc ID using recursive parsing for tables."""

    def _parse_elements(elements):
        """Internal helper to parse list of structural elements recursively."""
        text_output = []
        if not elements:
            return ""

        for value in elements:
            if 'paragraph' in value:
                p_elements = value.get('paragraph').get('elements')
                for elem in p_elements:
                    text_output.append(elem.get('textRun', {}).get('content', ''))
            elif 'table' in value:
                table = value.get('table')
                for row in table.get('tableRows'):
                    row_text = []
                    for cell in row.get('tableCells'):
                        # Recursive call to handle content inside cells (lists, etc.)
                        cell_content = _parse_elements(cell.get('content'))
                        # Clean up cell content (remove extra newlines)
                        row_text.append(cell_content.strip().replace("\n", " "))

                    # Create Markdown Table Row: | Col1 | Col2 |
                    text_output.append("| " + " | ".join(row_text) + " |\n")
        return "".join(text_output)

    try:
        service = build('docs', 'v1', credentials=creds)
        document = service.documents().get(documentId=doc_id).execute()
        doc_content = document.get('body').get('content')
        return _parse_elements(doc_content)
    except Exception as e:
        logger.error(f"❌ Error reading Doc {doc_id}: {e}")
        return ""

# --- B. MULTI-CACHE MANAGER ---
class SharedKnowledgeBase:
    """
    Manages multiple Vertex AI Context Caches.
    Creates a unique cache for every Agent that needs one.
    """
    def __init__(self, doc_mapping, sheet_mapping, creds):
        self.doc_mapping = doc_mapping
        self.sheet_mapping = sheet_mapping
        self.creds = creds
        self.cache_map = {} # Stores { "Framework_AgentName": CacheObject }
        self.common_docs_text = ""

    def _load_docs_once(self):
        """Reads docs only once to save time."""
        if self.common_docs_text: return

        logger.info(f"📚 Reading Knowledge Base (SOPs, T&Cs, DND List)...")
        # 1. Read Google Docs
        for name, doc_id in self.doc_mapping.items():
            content = get_google_doc_content(doc_id, self.creds)
            self.common_docs_text += f"\n\n=== DOCUMENT: {name} ===\n{content}\n"

        # 2. Read Google Sheets (NEW)
        for name, config in self.sheet_mapping.items():
            file_id = config.get('file_id')
            tabs = config.get('tabs', [])
            if file_id:
                content = get_google_sheet_content(file_id, tabs, self.creds)
                self.common_docs_text += f"\n\n=== SPREADSHEET: {name} ===\n{content}\n"

        if not self.common_docs_text.strip():
            logger.warning(f"⚠️ Warning: Knowledge Base is empty.")
            return None

    def create_agent_caches(self, framework_configs, model_name, ttl_hours=24):
        self._load_docs_once()
        if not self.common_docs_text.strip(): return

        logger.info(f"📦 Creating Dedicated Caches for Agents...")

        # Loop through all frameworks and agents
        for fw_name, fw_data in framework_configs.items():
            for agent in fw_data['agents']:
                if agent.get('use_cache', False):
                    cache_key = f"{fw_name}_{agent['name']}"
                    # 1. Prepare Specific System Instruction
                    # This is "baked" into the cache
                    full_system_prompt = agent['prompt']

                    try:
                        # 2. Create Cache
                        cache = caching.CachedContent.create(
                            model_name=model_name,
                            system_instruction=full_system_prompt,
                            contents=[Part.from_text(self.common_docs_text)],
                            ttl=datetime.timedelta(minutes=ttl_hours),
                        )
                        self.cache_map[cache_key] = cache
                        logger.info(f"✨ Cache Created: {cache_key}")
                    except Exception as e:
                        logger.error(f"❌ Failed to create cache for {cache_key}: {e}")

    def get_cache(self, framework_name, agent_name):
        return self.cache_map.get(f"{framework_name}_{agent_name}")

    def delete_all(self):
        logger.info(f"🗑️ Cleaning up caches...")
        for key, cache in self.cache_map.items():
            try:
                cache.delete()
                logger.info(f"🗑️ Deleted Cache: {key}")
            except: pass
        self.cache_map = {}
        logger.info(f"✅ All caches deleted.")

# --- A. MODEL REGISTRY (Singleton) ---
class ModelRegistry:
    """
    Ensures models are initialized only once per agent type/system prompt.
    Stores models in a dictionary: key=(agent_name, temperature, use_cache) -> value=GenerativeModel
    """
    _models = {}

    @classmethod
    def get_model(cls, agent_name, system_prompt, temperature, use_cache, cache_obj=None):
        # Unique key for this specific agent configuration
        key = (agent_name, temperature, use_cache)

        if key not in cls._models:
            logger.info(f"⚙️ Initializing Model for: {agent_name}")
            gen_config={
                        "temperature": temperature,
                        "response_mime_type": "text/plain" # Explicitly text
                        }

            # Initialize Model
            if use_cache and cache_obj:
                # BEST PRACTICE:
                # Initialize model FROM cache, but WITH specific system instructions.
                # This layers the Agent Persona on top of the Shared Data.
                # Note: We append the strict format template to the system prompt here.

                logger.info(f"🔗 Linking Cache to Agent: {agent_name}")
                model = GenerativeModel.from_cached_content(
                    cached_content=cache_obj,
                    generation_config=gen_config
                )
            else:
                # Standard Model (No Cache)
                model = GenerativeModel(
                    JOB_CONFIG['agent_model'],
                    system_instruction=system_prompt,
                    generation_config=gen_config
                )
            cls._models[key] = model

        return cls._models[key]

    @classmethod
    def clear(cls):
        cls._models = {}

# --- B. STRICT PARSERS ---
class StrictParser:
    """Parses text. Raises ValueError if format is wrong, triggering Retry."""

    @staticmethod
    def parse_sub_agent(text):
        """
        Expected: 'Category | L3 Value | Justification'
        Also accepts: 'None' (maps to None|None|None)
        """
        if not text: raise ValueError("Empty response")
        text = text.replace("```", "").strip()

        # 1. Handle the "None" case explicitly
        # If model outputs "None", "none", or "None."
        if text.lower().replace(".", "") == "none":
            return {
                "category": "None",
                "l3": "None",
                "justification": "None"
            }

        if text.count("|") != 2:
            text = "NA | " + text

        # 2. Regex: Capture 1 | Capture 2 | Capture 3
        # ^(.*?)\s*\|\s*(.*?)\s*\|\s*(.*)$
        match = re.search(r"^(.*?)\s*\|\s*(.*?)\s*\|\s*(.*)$", text, re.MULTILINE)

        if match:
            return {
                "category": match.group(1).strip(),
                "l3": match.group(2).strip(),
                "justification": match.group(3).strip()
            }

        # If the format doesn't match 3 parts with 2 pipes, raise Error
        raise ValueError(f"Invalid Format (Expected 'Cat|L3|Just'): {text[:50]}...")

    @staticmethod
    def parse_agent_4(text):
        """
        Expected: 'Primary L3 | Secondary L3'
        Handles:
        1. "L3 Value | None" -> Primary: L3, Secondary: None
        2. "L3 Value"        -> Primary: L3, Secondary: None (Implicit)
        3. Labels ("Primary:", "Secondary:") are stripped automatically to capture raw L3.
        """
        if not text:
            raise ValueError("Empty response")

        # 1. Remove Markdown
        text = text.replace("```", "").strip()

        # 2. Remove labels (Primary/Secondary) case-insensitively to get raw values
        # This fixes issues where model adds "Primary:" despite instructions.
        # It replaces "Primary:", "Primary -", "Secondary:", etc. with empty string.
        text = re.sub(r"(?i)\b(Primary|Secondary)\s*[:\-]?\s*", "", text)

        # 3. Logic: Check for Pipe Separator
        if "|" in text:
            parts = text.split("|", 1) # Split only on the first pipe found
            primary = parts[0].strip()
            # If right side is empty or whitespace, default to "None"
            secondary = parts[1].strip() if parts[1].strip() else "None"
        else:
            # 4. Fallback: No pipe found.
            # We assume the entire cleaned text is the Primary driver.
            # This ensures we don't lose the Primary L3 if the model forgot the secondary part.
            primary = text.strip()
            secondary = "None"

        # 5. Final Sanity Check
        # If primary became empty after cleaning (e.g., input was just "Primary:"), raise error to trigger Retry
        if not primary:
             raise ValueError(f"Invalid Format (Empty Primary): {text[:50]}")

        return {
            "primary": primary,
            "secondary": secondary
        }

# --- C. HIERARCHY ---
class HierarchyResolver:
    def __init__(self, hierarchy_dict):
        self.lookup_map = {}
        self._flatten_hierarchy(hierarchy_dict)

    def _flatten_hierarchy(self, hierarchy_dict):
        for fw_name, l1_dict in hierarchy_dict.items():
            for l1, l2_dict in l1_dict.items():
                for l2, l3_list in l2_dict.items():
                    for l3 in l3_list:
                        key = str(l3).strip().lower()
                        self.lookup_map[key] = {"L1": l1, "L2": l2, "L3": l3}

    def get_details(self, l3_value):
        key = str(l3_value).strip().lower()
        return self.lookup_map.get(key, {"L1": "Unmapped", "L2": "Unmapped", "L3": l3_value})

hierarchy_resolver = HierarchyResolver(HIERARCHY_CONFIG)

# --- D. DELTA MANAGER ---
class DeltaManager:
    @staticmethod
    def identify_delta(input_df, output_df):
        id_col = COL_MAPPING['case_number']
        status_col = COL_MAPPING['case_status']

        # Normalize IDs
        input_df[id_col] = input_df[id_col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

        if output_df is None or output_df.empty:
            return input_df, pd.DataFrame()

        output_df[id_col] = output_df[id_col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        output_df = output_df.drop_duplicates(subset=[id_col], keep='last')

        out_lookup = output_df.set_index(id_col)[status_col].to_dict()
        rows_to_process = []
        ids_to_process = set()

        for idx, row in input_df.iterrows():
            cid = row[id_col]
            status = str(row[status_col]).strip()

            # Logic: New Case OR Status Changed
            if cid not in out_lookup or status != str(out_lookup[cid]).strip():
                rows_to_process.append(idx)
                ids_to_process.add(cid)

        logger.info(f"🔍 Delta Analysis: Found {len(rows_to_process)} new/changed cases.")
        valid_output_df = output_df[~output_df[id_col].isin(ids_to_process)].copy()

        return input_df.loc[rows_to_process].copy(), valid_output_df


In [ ]:
import asyncio
from google.api_core.exceptions import ResourceExhausted, ServiceUnavailable, InternalServerError
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

# --- RETRY WRAPPER (Includes Generation AND Parsing) ---
@retry(
    wait=wait_exponential(multiplier=2, min=2, max=30),
    stop=stop_after_attempt(4),
    # Retry on Server Errors OR Parsing Errors (ValueError)
    retry=retry_if_exception_type((ResourceExhausted, ServiceUnavailable, InternalServerError, ValueError))
)
async def generate_and_parse(model, prompt, parser_func):
    """Generates content and immediately validates format. Retries if format is wrong."""
    response = await model.generate_content_async(prompt)
    # This line will raise ValueError if regex fails, triggering the @retry
    return parser_func(response.text)

# --- PIPELINE ---
async def run_framework_pipeline(row, framework_name, agent_configs, kb_manager):

    # 1. Build Input
    case_data = construct_universal_input(row, framework_name)

    # 1. Run Sub-Agents
    tasks = []
    for config in agent_configs:
        # Get Model from Registry (Initialized ONCE)
        agent_name = config['name']
        use_cache = config.get('use_cache', False)
        temp = config.get('temperature', JOB_CONFIG['model_temperature'])
        full_system_prompt = config['prompt']

        # Determine Cache Object
        cache_obj = None
        if use_cache:
            cache_obj = kb_manager.get_cache(framework_name, agent_name)

        # Get Model (Cache obj will have prompt baked in if exists)
        model = ModelRegistry.get_model(
            agent_name=f"{framework_name}_{agent_name}",
            system_prompt=full_system_prompt,
            use_cache=use_cache,
            cache_obj=cache_obj,
            temperature=temp
        )

        # Schedule Task (Gen + Parse) (Send only Case Data as User Input)
        tasks.append(generate_and_parse(model, case_data, StrictParser.parse_sub_agent))

    # Run in parallel
    results_list = await asyncio.gather(*tasks, return_exceptions=True)

    # Process Results & Handle Failures (if retries exhausted)
    agent_results = []
    for i, res in enumerate(results_list):
        if isinstance(res, Exception):
            # If after 4 retries it still fails
            agent_results.append({"category": "Error", "l3": "Error", "justification": str(res), "agent_name": agent_configs[i]['name']})
        else:
            res['agent_name'] = agent_configs[i]['name']
            agent_results.append(res)

    final_output = {}

    # 2. Logic: Quality/Workflow (Dynamic Drivers) - Ensure Unique L3s
    if framework_name in ["Quality", "Workflow"]:
        unique_l3s_added = set()
        driver_counter = 1
        for res in agent_results:
            l3_name = res['l3'].strip()
            # Only add if L3 is not 'None'/'Error' and not already added
            if l3_name.lower() not in ['none', 'error', 'unclear'] and l3_name not in unique_l3s_added:
                details = hierarchy_resolver.get_details(l3_name)
                final_output[f"driver_{driver_counter}"] = {
                    "L1": details['L1'], "L2": details['L2'], "L3": details['L3'],
                    "Justification": res['justification']
                }
                unique_l3s_added.add(l3_name)
                driver_counter += 1

    # 3. Logic: Others (Agent 4 Aggregation)
    else:
        # Findings for Agent 4
        # UPDATE: Format is now "Category | L3 | Justification"
        findings_list = []
        for r in agent_results:
            # Skip if L3 is None/Error to save tokens and confusion
            if str(r['l3']).lower() in ['none', 'error', 'unclear']:
                continue

            # Construct the line: "Category | L3 | Justification"
            # We sanitize pipes in the justification to prevent parsing confusion later (optional but safe)
            clean_just = str(r['justification']).replace("|", "-")
            findings_list.append(f"{r['category']} | {r['l3']} | {clean_just}")

        if not findings_list:
            # If sub-agents found nothing, skip Agent 4 call
            drivers = {"primary": "None", "secondary": "None"}
        else:
            findings_text = "\n".join(findings_list)

            # Get Model for Agent 4
            model_a4 = ModelRegistry.get_model(
                agent_name="Agent_4_Aggregator",
                system_prompt=HIERARCHY_AGENT_4_PROMPT,
                use_cache=False,
                cache_obj=None,
                temperature = 0.2
            )

            # Generate & Parse Agent 4
            try:
                drivers = await generate_and_parse(model_a4, findings_text, StrictParser.parse_agent_4)
            except Exception as e:
                # Log error and default to None
                # print(f"Agent 4 Failed: {e}")
                # log the error with appropriate symbol or emoji
                logger.error(f" Agent 4 Failed: {e}")
                drivers = {"primary": "Error", "secondary": "Error"}

        # Map Justifications
        # We look up the original justification using the L3 name returned by Agent 4
        just_map = {str(r['l3']).lower().strip(): r['justification'] for r in agent_results}

        p_l3 = drivers['primary']
        s_l3 = drivers['secondary']

        p_det = hierarchy_resolver.get_details(p_l3)
        s_det = hierarchy_resolver.get_details(s_l3)

        # --- WHITESPACE NORMALIZATION ---
        # Strip whitespace from L1 values before comparing
        p_l1_clean = str(p_det.get('L1', '')).strip()
        s_l1_clean = str(s_det.get('L1', '')).strip()

        # 1. Always add Driver 1
        final_output = {
            "driver_1": {
                "L1": p_det['L1'], "L2": p_det['L2'], "L3": p_det['L3'],
                "Justification": just_map.get(str(p_l3).lower().strip(), "None")
            }
        }

        # 2. Add Driver 2 ONLY if L1 is different AND it's not None/Error
        # Comparison now uses the cleaned (stripped) L1 strings
        is_different_l1 = (s_l1_clean != p_l1_clean)
        is_valid_l3 = (str(s_l3).lower().strip() not in ['none', 'error', 'unmapped'])

        if is_different_l1 and is_valid_l3:
            final_output["driver_2"] = {
                "L1": s_det['L1'], "L2": s_det['L2'], "L3": s_det['L3'],
                "Justification": just_map.get(str(s_l3).lower().strip(), "None")
            }

    return json.dumps(final_output)


async def process_single_row(row, semaphore, cache_obj):
    async with semaphore:
        pipeline_tasks = {}

        # Conditionals
        if pd.to_numeric(row.get(COL_MAPPING['reopen_counter'], 0), errors='coerce') > 0:
            pipeline_tasks['out_reopen'] = asyncio.create_task(run_framework_pipeline(row, "Reopen", FRAMEWORK_CONFIGS['Reopen']['agents'], cache_obj))

        if pd.to_numeric(row.get(COL_MAPPING['aging_days'], 0), errors='coerce') > 5:
            pipeline_tasks['out_ttr'] = asyncio.create_task(run_framework_pipeline(row, "TTR", FRAMEWORK_CONFIGS['TTR']['agents'], cache_obj))

        if str(row.get(COL_MAPPING['is_escalated'], '')).upper() == 'TRUE':
            pipeline_tasks['out_escalation'] = asyncio.create_task(run_framework_pipeline(row, "Escalation", FRAMEWORK_CONFIGS['Escalation']['agents'], cache_obj))

        if str(row.get(COL_MAPPING['survey_result'], '')).upper() == 'DSAT':
            pipeline_tasks['out_dsat'] = asyncio.create_task(run_framework_pipeline(row, "DSAT", FRAMEWORK_CONFIGS['DSAT']['agents'], cache_obj))

        #Always Run Quality/Workflow
        pipeline_tasks['out_quality'] = asyncio.create_task(run_framework_pipeline(row, "Quality", FRAMEWORK_CONFIGS['Quality']['agents'], cache_obj))
        pipeline_tasks['out_workflow'] = asyncio.create_task(run_framework_pipeline(row, "Workflow", FRAMEWORK_CONFIGS['Workflow']['agents'], cache_obj))

        # Gather
        results_map = {}
        if pipeline_tasks:
            keys = list(pipeline_tasks.keys())
            gathered = await asyncio.gather(*pipeline_tasks.values())
            for k, v in zip(keys, gathered):
                results_map[k] = v

        results_map['out_timestamp'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return results_map



In [ ]:
# @title 10. Main Execution

async def run_full_job():
    ModelRegistry.clear()
    kb_manager = SharedKnowledgeBase(DOC_MAPPING, SHEET_MAPPING, creds)

    try:
        logger.info(f"📥 Loading Data from Sheets...")
        spread = Spread(IO_CONFIG['spreadsheet_id'], creds=creds)
        input_df = spread.sheet_to_df(index=0, sheet=IO_CONFIG['input_sheet_name'])
        #input_df = input_df.iloc[:700].copy()
        # Load Output Sheet to check for existing rows
        try:
            output_df = spread.sheet_to_df(index=0, sheet=IO_CONFIG['output_sheet_name'])
        except:
            output_df = pd.DataFrame(columns=OUTPUT_COLS_TO_SAVE)

        # 1. Identify New/Changed Cases
        delta_df, valid_output_df = DeltaManager.identify_delta(input_df, output_df)

        if delta_df.empty:
            logger.info("✅ No new or changed cases to process. Job Finished.")
            return

        logger.info(f"🚀 Processing {len(delta_df)} cases in chunks of {JOB_CONFIG['save_chunk_size']}...")

        # 2. RESTORE VALID HISTORY (The logic I missed)
        # We overwrite the sheet with the valid history first.
        # This ensures we don't lose the data we are NOT processing.

        # Ensure valid_df has correct columns
        for col in OUTPUT_COLS_TO_SAVE:
            if col not in valid_output_df.columns:
                valid_output_df[col] = ""
        valid_output_df = valid_output_df[OUTPUT_COLS_TO_SAVE]

        logger.info(f"💾 Restoring {len(valid_output_df)} valid historical rows...")
        spread.df_to_sheet(
            valid_output_df,
            index=False,
            headers=True,
            sheet=IO_CONFIG['output_sheet_name'],
            replace=True # Overwrite to set the clean state
        )

        # 3. CALCULATE START ROW FOR APPENDING
        # If valid_df has 100 rows, Sheet has 101 rows (1 Header + 100 Data).
        # We start writing at Row 102.
        # If valid_df is empty, Sheet has 0 rows. We start at Row 1 (and need to write header).

        existing_count = len(valid_output_df)
        start_row_index = existing_count + 2 if existing_count > 0 else 1

        # Create Caches ONCE
        kb_manager.create_agent_caches(FRAMEWORK_CONFIGS, model_name=JOB_CONFIG['agent_model'])

        # 4. CHUNK PROCESSING LOOP
        total_rows = len(delta_df)
        chunk_size = JOB_CONFIG['save_chunk_size']

        for i in range(0, total_rows, chunk_size):
            chunk_df = delta_df.iloc[i : i + chunk_size].copy()
            logger.info(f"🔄 Processing Chunk {i} to {min(i+chunk_size, total_rows)}...")

            # A. Process Async
            sem = asyncio.Semaphore(JOB_CONFIG['batch_size'])
            tasks = [process_single_row(row, sem, kb_manager) for _, row in chunk_df.iterrows()]
            results = await tqdm.gather(*tasks)

            # B. Merge Results
            for idx, (df_index, row) in enumerate(chunk_df.iterrows()):
                res = results[idx]
                for col_key, df_col in COL_MAPPING.items():
                    if col_key.startswith("out_"):
                        val = res.get(col_key, "")
                        chunk_df.at[df_index, df_col] = val


            # --- NEW: Calculate L3 Score and Grade ---
            l3_scores_for_chunk = []
            grades_for_chunk = []

            for _, row in chunk_df.iterrows():
                total_row_score = 0
                # Use a set to track processed L3s to avoid double counting if an L3 appears across different agent outputs
                processed_l3s = set()

                # Iterate through all output columns that contain agent results (e.g., Quality_RCA_Output, Workflow_RCA_Output)
                # We need to exclude 'out_timestamp' and the new 'out_score'/'out_grade' itself from this loop
                agent_output_cols_to_process = [
                    COL_MAPPING["out_quality"]
                    # Removed COL_MAPPING["out_workflow"] as per user's request.
                    # Add other framework output columns here if they are enabled (e.g., COL_MAPPING["out_reopen"])
                ]

                for output_col_name in agent_output_cols_to_process:
                    # Check if the column exists in the row and has a non-empty value
                    if output_col_name in row and row[output_col_name] and pd.notna(row[output_col_name]):
                        try:
                            # Parse the JSON string from the agent's output
                            l3_violations_dict = json.loads(row[output_col_name])

                            # The structure for Quality/Workflow is direct L1/L2/L3 in driver_1, driver_2, etc.
                            # So iterate through these drivers
                            for driver_key in ['driver_1', 'driver_2', 'driver_3','driver_4','driver_5']: # Assuming max 3 drivers per framework output
                                if driver_key in l3_violations_dict and isinstance(l3_violations_dict[driver_key], dict):
                                    violation_entry = l3_violations_dict[driver_key]
                                    l3_name = violation_entry.get('L3')

                                    if l3_name and l3_name not in processed_l3s: # Check if L3 has not been scored yet
                                        # Only add score if it's a valid L3 (not 'None' or 'Error' as reported by agent)
                                        if str(l3_name).lower() not in ['none', 'error']:
                                            # Get score from SCORING_DICT, default to 0 if L3 is not found
                                            score_value = SCORING_DICT.get(l3_name, 0)
                                            total_row_score += score_value
                                            processed_l3s.add(l3_name) # Add to set to prevent double counting
                        except json.JSONDecodeError as e:
                            logger.error(f"[{JOB_ID}] JSON parsing error for case {row.get(COL_MAPPING['case_number'], 'N/A')} in column '{output_col_name}': {e}")
                        except Exception as e:
                            logger.error(f"[{JOB_ID}] General error processing L3 score for case {row.get(COL_MAPPING['case_number'], 'N/A')} from '{output_col_name}': {e}")

                l3_scores_for_chunk.append(total_row_score)
                # Determine Grade: 'Fail' if total_row_score is greater than a threshold (e.g., 0 for any violation), else 'Pass'
                # The user's snippet implied (100 - x) <= 90 which means x > 10 is 'Fail'.
                # Let's use a simple threshold for demonstration; e.g., any score above 0 is a 'Fail' if scores represent penalties.
                grades_for_chunk.append('Fail' if total_row_score < 90 else 'Pass')

            chunk_df[COL_MAPPING['out_score']] = l3_scores_for_chunk
            chunk_df[COL_MAPPING['out_grade']] = grades_for_chunk

            # --- END NEW CODE ---
            # C. Prepare for Save
            for col in OUTPUT_COLS_TO_SAVE:
                if col not in chunk_df.columns:
                    chunk_df[col] = ""
            chunk_to_save = chunk_df[OUTPUT_COLS_TO_SAVE]

            # D. Save Chunk (Append Mode)
            try:
                # Logic: We need a header ONLY if we are at Row 1 (i.e., valid_output_df was empty)
                # AND this is the very first chunk.
                write_header = True if start_row_index == 1 else False

                spread.df_to_sheet(
                    chunk_to_save,
                    index=False,
                    headers=write_header, # Corrected argument name
                    sheet=IO_CONFIG['output_sheet_name'],
                    start=(start_row_index, 1),
                    replace=False
                )
                logger.info(f"✅ Chunk saved at Row {start_row_index}")

                # E. Update Pointer
                # If we wrote header, we used (Len + 1) rows. Else (Len) rows.
                rows_occupied = len(chunk_to_save) + (1 if write_header else 0)
                start_row_index += rows_occupied

            except Exception as save_err:
                logger.error(f"❌ SHEET SAVE FAILED: {save_err}")
                backup_name = f"backup_chunk_{i}.csv"
                chunk_to_save.to_csv(backup_name, index=False)
                logger.warning(f"⚠️ Saved to local file '{backup_name}' instead.")


    except Exception as e:
        logger.critical(f"💀 FATAL JOB ERROR: {e}")
        import traceback
        logger.error(traceback.format_exc())

    finally:
        kb_manager.delete_all()
        logger.info("🏁 Job Complete.")


In [ ]:
if __name__ == "__main__":
    await run_full_job()

INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:📥 Loading Data from Sheets...
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:🚀 Processing 19666 cases in chunks of 50...
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:💾 Restoring 0 valid historical rows...
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:📚 Reading Knowledge Base (SOPs, T&Cs, DND List)...
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:📦 Creating Dedicated Caches for Agents...
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:✨ Cache Created: Reopen_Process_Knowledge_Auditor
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:✨ Cache Created: TTR_Process_RootCause_Auditor
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:✨ Cache Created: Escalation_Policy_RAG
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:✨ Cache Created: DSAT_Policy_Process_Expert
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:✨ Cache Created: Quality_Compliance_1
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:✨ Cache Created: Workflow_review
INFO:GTM_Rubrics_RCA_2026_03_10_13_03_32:🔄 Processing Chunk 0 to 50...
  0%|          | 0/50